# Topic Modeling Workflow: Representative Selected Notes


## Outputs

This notebook writes the canonical Representative topic artifacts under
`data/processed/topics/` (or `.artifacts/smoke/data/processed/topics/`).


In [ ]:
from pathlib import Path
import sys

def find_project_root() -> Path:
    """Anchor on config.txt — safe from any subdir depth."""
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "config.txt").exists():
            return p
    raise RuntimeError("config.txt not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from src.config import TopicConfig
from src.io import get_interim_dir, get_processed_dir, get_test_mode, get_topic_dir, load_table, load_ratings_with_final_cluster, save_table
from src.topics import (
    _EXTRA_STOP_WORDS,
    add_topic_display_labels,
    attach_topic_names,
    build_salience_pivot,
    build_strategy_topic_pivot,
    build_topic_cluster_stats,
    build_topic_exemplars,
    build_topic_rescue_stats,
    build_topic_salience,
    build_topic_selection_overlap,
    build_topic_strategy_summary,
    fit_topic_model,
    get_cluster_approval_columns,
    prepare_topic_frame,
)

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 1000)
sns.set_theme(style='whitegrid', context='notebook')

TEST_MODE = get_test_mode()
print(f'=== TEST_MODE = {TEST_MODE}'
      f' (' + ('smoke test (small sample)' if TEST_MODE else 'FULL DATA') + ') ===')


def topic_label_column(df: pd.DataFrame) -> str:
    return 'topic_display_label' if 'topic_display_label' in df.columns else 'topic_label'


def avg_cluster_columns(df: pd.DataFrame) -> list[str]:
    return sorted(
        [col for col in df.columns if col.startswith('avg_cluster_')],
        key=lambda col: int(col.rsplit('_', 1)[1]),
    )


def cluster_id_from_avg_col(col: str) -> int:
    return int(col.rsplit('_', 1)[1])


def ordered_unique(values) -> list:
    return list(dict.fromkeys(list(values)))


## Configuration

In [ ]:
topic_config = TopicConfig(
    embedding_model_name='all-MiniLM-L6-v2',
    random_state=42,
    top_k_topics=10,
    top_k_exemplars=5,
    salience_top_n=30,
)

INTERIM_DIR = get_interim_dir()
SCORING_DIR = get_processed_dir()
PROCESSED_DIR = get_topic_dir()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f'Topic outputs will be written to: {PROCESSED_DIR}')


## Load Upstream Inputs

The topic workflow now also pulls in the scoring-stage selection logs. That lets topic analysis answer not only where disagreement lives, but also which topics feed the NMR rescue problem and which topics are preferentially surfaced by the Simple Majoritarian Rule, pluralistic, and representative selection rules.


In [ ]:
scores = load_table(SCORING_DIR / 'scores.parquet')
ratings_clustered = load_ratings_with_final_cluster()
selection_log = load_table(SCORING_DIR / 'selection_log.parquet')
selection_status_summary = load_table(SCORING_DIR / 'selection_status_summary.parquet')

print(f'Scored notes available for topic modeling: {len(scores):,}')
print(f'Clustered rating rows available for salience analysis: {len(ratings_clustered):,}')
print(f'Selection log rows available for strategy-aware topic analysis: {len(selection_log):,}')
print(f'Strategy status summary rows: {len(selection_status_summary):,}')

## Prepare Note-Level Topic Input

This table is intentionally narrow: one row per note, the note text, cluster approvals, vote volume, and disagreement features. That keeps topic modeling focused and makes the output easier to reuse later.

In [ ]:
# Topic modeling input = all notes selected by the Representative strategy.
# This intentionally does not use diagnostic_notes.parquet: every Representative
# selected note present in scores.parquet is included.
representative_ids = set(
    selection_log.loc[
        selection_log['strategy'].eq('Representative'),
        'selected_noteId',
    ].dropna().astype(str)
)
scores_for_topics = scores[scores['noteId'].astype(str).isin(representative_ids)].copy()
missing_representative_ids = representative_ids.difference(set(scores['noteId'].astype(str)))
print(
    f'Representative topic input: {len(scores_for_topics):,} scored notes matched '
    f'from {len(representative_ids):,} Representative selected noteIds.'
)
print(f'Representative selected noteIds missing from scores.parquet: {len(missing_representative_ids):,}')

topic_input = prepare_topic_frame(scores_for_topics)
display(topic_input.head())
print('Topic input rows:', len(topic_input))

## Fit BERTopic

This is the most expensive stage. It is separated on purpose so you can rerun topic work without touching clustering or rescue logic.

In [ ]:
topic_model, topic_notes = fit_topic_model(
    topic_input,
    embedding_model_name=topic_config.embedding_model_name,
    random_state=topic_config.random_state,
)

topic_notes = attach_topic_names(topic_model, topic_notes)
topic_notes = add_topic_display_labels(topic_notes)
display(topic_notes.head())

## Topic Label Cleaning Audit

BERTopic topic names can become visually unusable because they often contain raw URL fragments, long slugs, or over-specific token chains. To keep the analysis interpretable, this workflow creates a cleaned `topic_label` alongside the original `Name`.

Keep this audit table in the notebook so you can always inspect what was shortened and decide whether any topic needs a manual rename before you move on to qualitative interpretation or figure drafting.

In [ ]:
topic_label_audit = (
    topic_notes[['topic', 'Name', 'topic_label', 'topic_display_label']]
    .drop_duplicates(['topic', 'Name', 'topic_label', 'topic_display_label'])
    .sort_values('topic')
    .reset_index(drop=True)
)

display(topic_label_audit.head(30))


## Topic-Level Disagreement Summary

This keeps the strongest insight from the original pipeline: some topics are much more polarized across clusters than others, and those high-gap topics deserve focused qualitative inspection.

In [ ]:
topic_cluster_stats = build_topic_cluster_stats(topic_notes)
topic_cluster_stats = add_topic_display_labels(topic_cluster_stats)
exemplars = build_topic_exemplars(
    topic_notes,
    topic_cluster_stats,
    top_k_topics=topic_config.top_k_topics,
    top_k_exemplars=topic_config.top_k_exemplars,
)
exemplars = add_topic_display_labels(exemplars)

summary_cols = [
    'topic', 'topic_label', 'topic_display_label', 'notes', 'avg_votes',
    'approval_range', 'abs_gap', 'top_cluster', 'bottom_cluster',
]
summary_cols = [col for col in summary_cols if col in topic_cluster_stats.columns]
display(topic_cluster_stats[summary_cols].head(15))

exemplar_cols = ['topic', 'topic_display_label', 'Name', 'summary', 'abs_gap', 'total_votes']
exemplar_cols = [col for col in exemplar_cols if col in exemplars.columns]
display(exemplars[exemplar_cols].head(20))


## Topic Salience by Cluster

The original notebook explored whether some clusters spend more attention on some topics. This section keeps that idea but turns it into reusable tables instead of one-off cells.

In [ ]:
note_topic_cols = ['noteId', 'topic', 'Name', 'topic_label']
if 'topic_display_label' in topic_notes.columns:
    note_topic_cols.append('topic_display_label')
note_topic_map = topic_notes[note_topic_cols].drop_duplicates('noteId')

topic_salience = build_topic_salience(ratings_clustered, note_topic_map)
topic_salience = add_topic_display_labels(topic_salience)
topic_salience_pivot = build_salience_pivot(topic_salience)
topic_salience_pivot = add_topic_display_labels(topic_salience_pivot)

display_cols = ['topic', topic_label_column(topic_salience), 'cluster', 'total_ratings', 'positive_rate']
display(topic_salience[display_cols].head())
display(topic_salience_pivot.head(15))


## Topic x Rescue Analysis

This table is especially useful for the next research phase. It highlights topics where there seems to be consensus under the representative score but the platform still suppresses visibility.

In [ ]:
topic_rescue_stats = build_topic_rescue_stats(scores, note_topic_map, bridge_threshold=0.25)
topic_rescue_stats = add_topic_display_labels(topic_rescue_stats)
rescue_cols = ['topic', topic_label_column(topic_rescue_stats), 'consensus_notes', 'consensus_failures', 'failure_rate_within_consensus']
display(topic_rescue_stats[rescue_cols].head(15))


## Strategy-Aware Topic Analysis

This layer connects the new scoring outputs back into topic space. Instead of only asking which topics are polarized, we now also ask:
- which topics each strategy tends to select
- whether those selected notes are mostly already `Helpful` or still `NMR`
- where representative or pluralistic selection diverges from the Simple Majoritarian Rule inside each topic


In [ ]:
topic_note_cols = ['noteId', 'topic', 'Name', 'topic_label']
if 'topic_display_label' in topic_notes.columns:
    topic_note_cols.append('topic_display_label')
topic_note_map = topic_notes[topic_note_cols].drop_duplicates('noteId')

topic_strategy_summary = build_topic_strategy_summary(selection_log, topic_note_map)
topic_strategy_summary = add_topic_display_labels(topic_strategy_summary)
topic_strategy_pivot = build_strategy_topic_pivot(topic_strategy_summary, value_col='selected_picks')
topic_strategy_pivot = add_topic_display_labels(topic_strategy_pivot)
topic_selection_overlap = build_topic_selection_overlap(selection_log, topic_note_map, compare_to='Simple Majoritarian Rule')
topic_selection_overlap = add_topic_display_labels(topic_selection_overlap)

strategy_cols = ['strategy', topic_label_column(topic_strategy_summary), 'status_group', 'selected_picks', 'unique_tweets']
display(topic_strategy_summary[strategy_cols].head(20))
display(topic_strategy_pivot.head(15))
overlap_cols = [
    col
    for col in ['strategy', topic_label_column(topic_selection_overlap), 'overlap_rate', 'matching_rows', 'total_rows']
    if col in topic_selection_overlap.columns
]
display(topic_selection_overlap[overlap_cols].head(20))


## Topic Diagnostics

These topic diagnostics now focus entirely on representative selection and disagreement, without the consensus-break scatter.

Read them as follows:
- the first chart shows the highest-disagreement topics and how many representative picks each one actually receives
- the second chart shows which topics the representative rule selects most often, split by `Helpful` vs `NMR`
- the third chart restricts attention to high-disagreement topics that were selected by the representative rule and shows their status composition

In [ ]:
rep_topic_status = (
    topic_strategy_summary[topic_strategy_summary['strategy'] == 'Representative']
    .copy()
)
label_col = topic_label_column(topic_strategy_summary)
stats_label_col = topic_label_column(topic_cluster_stats)

rep_topic_totals = (
    rep_topic_status.groupby(['topic', label_col, 'status_group'])['selected_picks']
    .sum()
    .reset_index()
)
rep_topic_totals = rep_topic_totals.rename(columns={label_col: 'topic_plot_label'})

rep_topic_presence = (
    rep_topic_status.groupby(['topic', label_col])['selected_picks']
    .sum()
    .reset_index(name='representative_selected_picks')
    .rename(columns={label_col: 'topic_plot_label'})
)

topic_stats_for_plot = topic_cluster_stats[['topic', stats_label_col, 'abs_gap']].rename(
    columns={stats_label_col: 'topic_plot_label'}
)

# 1) Highest-disagreement topics with representative selection intensity
plot_df = (
    topic_stats_for_plot
    .merge(rep_topic_presence, on=['topic', 'topic_plot_label'], how='left')
    .fillna({'representative_selected_picks': 0})
    .sort_values('abs_gap', ascending=False)
    .head(15)
    .sort_values('abs_gap', ascending=True)
)

plt.figure(figsize=(12, 8))
ax = sns.barplot(
    data=plot_df,
    x='abs_gap',
    y='topic_plot_label',
    color='#cbd5e1',
)
for row in plot_df.itertuples(index=False):
    ax.text(
        row.abs_gap + 0.01,
        row.topic_plot_label,
        f"Rep picks: {int(row.representative_selected_picks)}",
        va='center',
        fontsize=10,
        color='#065f46',
        fontweight='bold',
    )
ax.set_title('Highest-Disagreement Topics with Representative Pick Counts', fontsize=17, fontweight='bold', pad=14)
ax.set_xlabel('Approval range across clusters (max − min)')
ax.set_ylabel('Topic')
plt.tight_layout()
plt.show()

# 2) Representative most frequently selected topics, split by current status
rep_top_topics = (
    rep_topic_totals.groupby(['topic', 'topic_plot_label'])['selected_picks']
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)
rep_plot = rep_topic_totals.merge(rep_top_topics[['topic', 'topic_plot_label']], on=['topic', 'topic_plot_label'], how='inner')
topic_order = ordered_unique(rep_top_topics.sort_values('selected_picks', ascending=True)['topic_plot_label'])
rep_plot['topic_plot_label'] = pd.Categorical(rep_plot['topic_plot_label'], categories=topic_order, ordered=True)
rep_plot = rep_plot.sort_values('topic_plot_label')

plt.figure(figsize=(12, 8))
sns.barplot(
    data=rep_plot,
    x='selected_picks',
    y='topic_plot_label',
    hue='status_group',
    palette={'Helpful': '#0f766e', 'NMR': '#dc2626', 'Other': '#9ca3af'},
)
plt.title('Most Frequently Selected Topics by Representative Rule', fontsize=17, fontweight='bold', pad=14)
plt.xlabel('Representative selected picks')
plt.ylabel('Topic')
plt.legend(frameon=False, title='Current status')
plt.tight_layout()
plt.show()

# 3) High-disagreement topics selected by representative rule, split by status
rep_high_gap = topic_stats_for_plot.merge(rep_topic_totals, on=['topic', 'topic_plot_label'], how='inner')
rep_high_gap = rep_high_gap[rep_high_gap['selected_picks'] > 0].copy()
rep_high_gap = rep_high_gap.sort_values('abs_gap', ascending=False)

top_rep_high_gap_topics = rep_high_gap[['topic', 'topic_plot_label', 'abs_gap']].drop_duplicates(['topic', 'topic_plot_label']).head(15)
rep_high_gap = rep_high_gap.merge(top_rep_high_gap_topics[['topic', 'topic_plot_label']], on=['topic', 'topic_plot_label'], how='inner')
topic_order = ordered_unique(top_rep_high_gap_topics.sort_values('abs_gap', ascending=True)['topic_plot_label'])
rep_high_gap['topic_plot_label'] = pd.Categorical(rep_high_gap['topic_plot_label'], categories=topic_order, ordered=True)
rep_high_gap = rep_high_gap.sort_values('topic_plot_label')

plt.figure(figsize=(12, 8))
sns.barplot(
    data=rep_high_gap,
    x='selected_picks',
    y='topic_plot_label',
    hue='status_group',
    palette={'Helpful': '#0f766e', 'NMR': '#dc2626', 'Other': '#9ca3af'},
)
plt.title('High-Disagreement Topics Selected by Representative Rule', fontsize=17, fontweight='bold', pad=14)
plt.xlabel('Representative selected picks')
plt.ylabel('Topic')
plt.legend(frameon=False, title='Current status')
plt.tight_layout()
plt.show()


## A1 — Cluster-Based Note Text Analysis (TF-IDF)

So far we only knew the clusters differ behaviorally. This section asks: *what kind of content does each cluster endorse?* Notes with strong cluster-specific approval (`cluster_X_approval ≥ 0.8`) are collected per cluster, TF-IDF is fitted on the combined corpus, and differential term weights are compared. Stop-word list is the union of scikit-learn English stops and the project-level extra stops from `src/topics.py` (Spanish function words, URL fragments, Twitter noise).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text as sk_text
from src.topics import _EXTRA_STOP_WORDS as _TOPIC_EXTRA_STOPS

# stop_tokens used inside shorten_topic_name — not exported, so we mirror them here
_TOPIC_NAME_STOPS = {
    "http", "https", "html", "www", "com", "news", "story",
    "article", "read", "nnn", "cn", "que",
}

_CN_NOISE = {
    "sns", "nhk", "post", "posts", "opinion", "note", "tweet", "tweets",
    "community", "notes", "context", "claim", "claims", "source",
    "sources", "video", "image", "photo", "says", "said", "stated",
    "states", "added", "wrote", "writing",
}

_ALL_STOP_WORDS = list(
    sk_text.ENGLISH_STOP_WORDS
    .union(_TOPIC_EXTRA_STOPS)
    .union(_TOPIC_NAME_STOPS)
    .union(_CN_NOISE)
)

HIGH_APPROVAL_THRESHOLD = 0.8
TFIDF_TOP_N = 20
cluster_cols = get_cluster_approval_columns(topic_notes)
cluster_ids = [int(col.split('_')[1]) for col in cluster_cols]

corpus = []
labels = []
for cluster_id, col in zip(cluster_ids, cluster_cols):
    cluster_corpus = topic_notes[topic_notes[col] >= HIGH_APPROVAL_THRESHOLD]['summary'].dropna().astype(str).tolist()
    print(f"Cluster {cluster_id} high-approval notes: {len(cluster_corpus):,}  ({col} >= {HIGH_APPROVAL_THRESHOLD})")
    corpus.extend(cluster_corpus)
    labels.extend([cluster_id] * len(cluster_corpus))

if not corpus or len(set(labels)) < 2:
    print('Not enough high-approval corpora to compute cluster-level TF-IDF comparison.')
else:
    vectorizer = TfidfVectorizer(
        max_features=8000,
        ngram_range=(1, 2),
        stop_words=_ALL_STOP_WORDS,
        min_df=3,
        max_df=0.85,
    )
    tfidf_matrix = vectorizer.fit_transform(corpus)
    feature_names = vectorizer.get_feature_names_out()

    mean_by_cluster = {}
    for cluster_id in cluster_ids:
        idx = [i for i, label in enumerate(labels) if label == cluster_id]
        if idx:
            mean_by_cluster[cluster_id] = np.asarray(tfidf_matrix[idx].mean(axis=0)).ravel()

    n_clusters = len(mean_by_cluster)
    fig, axes = plt.subplots(1, n_clusters, figsize=(6 * n_clusters, 7), squeeze=False)
    axes = axes.ravel()
    colors = ['#D55E00', '#0072B2', '#009E73', '#CC79A7', '#E69F00']

    for ax, cluster_id in zip(axes, mean_by_cluster):
        current = mean_by_cluster[cluster_id]
        others = [vec for cid, vec in mean_by_cluster.items() if cid != cluster_id]
        other_mean = np.vstack(others).mean(axis=0)
        diff = current - other_mean
        top_idx = np.argsort(diff)[::-1][:TFIDF_TOP_N]
        df_terms = pd.DataFrame({
            'term': feature_names[top_idx],
            'differential': diff[top_idx],
        }).sort_values('differential')
        display(df_terms.sort_values('differential', ascending=False).head(15))
        ax.barh(df_terms['term'], df_terms['differential'], color=colors[cluster_id % len(colors)], alpha=0.85)
        ax.axvline(0, color='black', linewidth=0.7, linestyle='--')
        ax.set_title(f'Cluster {cluster_id} — Distinctive Terms\n(vs. other clusters)')
        ax.set_xlabel('Differential TF-IDF')

    fig.suptitle('Cluster-Level TF-IDF Comparison — What Content Does Each Cluster Endorse?', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


## A2 — Systematizing Disagreement Direction

Figure 4 only covered the ten most polarized topics. This section extends to all topics and asks: *which cluster is systematically more positive?*

- If the same cluster is most positive in the large majority of topics, that suggests a **lenient vs. strict** split.
- If the most-positive cluster changes across topics, that suggests a **partisan/content-dependent** split.
- With three or more clusters, the middle cluster is interpreted through where it sits between the least- and most-approving blocs.

In [ ]:
stats = topic_cluster_stats.copy()
stats = stats[stats['topic'] != -1].copy()  # exclude noise topic
label_col = topic_label_column(stats)
avg_cols = avg_cluster_columns(stats)

if len(avg_cols) < 2:
    print('Need at least two clusters for disagreement direction diagnostics.')
else:
    values = stats[avg_cols].astype(float)
    stats['approval_min'] = values.min(axis=1)
    stats['approval_max'] = values.max(axis=1)
    stats['approval_range'] = stats['approval_max'] - stats['approval_min']
    stats['abs_gap'] = stats['approval_range']
    stats['top_cluster'] = values.idxmax(axis=1).map(cluster_id_from_avg_col)
    stats['bottom_cluster'] = values.idxmin(axis=1).map(cluster_id_from_avg_col)
    stats['favored_cluster'] = stats['top_cluster'].map(lambda cid: f'Cluster {cid} leads')

    direction_counts = stats['favored_cluster'].value_counts()
    print('Which cluster is most positive across all topics?')
    print(direction_counts.to_string())
    for label, count in direction_counts.items():
        print(f"{label}: {count} / {len(stats)} topics ({count/len(stats):.1%})")

    high_gap = stats[stats['abs_gap'] > 0.3].copy()
    print(f"\nAmong high-disagreement topics (approval range > 0.3, n={len(high_gap)}):")
    print(high_gap['favored_cluster'].value_counts().to_string())

    top50 = stats.nlargest(50, 'abs_gap').sort_values('abs_gap', ascending=True)
    colors = ['#D55E00', '#0072B2', '#009E73', '#CC79A7', '#E69F00']
    bar_colors = [colors[int(cid) % len(colors)] for cid in top50['top_cluster']]

    fig, ax = plt.subplots(figsize=(10, 16))
    ax.barh(top50[label_col], top50['abs_gap'], color=bar_colors, alpha=0.85)
    ax.set_title('Top 50 Topics by Cross-Cluster Approval Range', fontsize=13, fontweight='bold')
    ax.set_xlabel('Approval range across clusters (max − min)')
    ax.set_ylabel('Topic')
    ax.tick_params(axis='y', labelsize=8.5)
    legend_handles = [Line2D([0], [0], color=colors[int(cid) % len(colors)], lw=6, label=f'Cluster {cid} highest') for cid in sorted(stats['top_cluster'].dropna().unique())]
    ax.legend(handles=legend_handles, frameon=False, loc='lower right', fontsize=8.5)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 7))
    sc = ax.scatter(
        stats['approval_min'],
        stats['approval_max'],
        c=stats['top_cluster'],
        cmap='tab10',
        s=stats['notes'] * 4,
        alpha=0.65,
        edgecolors='white', linewidths=0.4,
    )
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='No disagreement')
    for _, row in stats[stats['abs_gap'] > 0.70].iterrows():
        ax.annotate(
            row[label_col][:28],
            (row['approval_min'], row['approval_max']),
            fontsize=7, alpha=0.85,
            xytext=(5, 5), textcoords='offset points',
        )
    plt.colorbar(sc, ax=ax, label='Cluster with highest mean approval')
    ax.set_title('Per-Topic Cross-Cluster Approval Range\n(dot size proportional to note count)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Lowest cluster mean approval')
    ax.set_ylabel('Highest cluster mean approval')
    ax.legend(loc='upper left', frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()


## A3 — Cluster-Level User Profiling

Additional behavioral statistics to characterize *who* is in each cluster. Per-user total vote count and positive vote rate are computed from `ratings_clustered`; cluster-level distributions are compared. Enriches Table 1 and helps distinguish "active vs. passive" or "lenient vs. strict" cluster profiles.

In [ ]:
# Per-user behavioral stats from ratings_clustered
user_stats_raw = (
    ratings_clustered
    .groupby(['raterParticipantId', 'cluster'])
    .agg(
        total_votes=('vote', 'count'),
        positive_votes=('vote', 'sum'),
    )
    .reset_index()
)
user_stats_raw['positive_rate'] = user_stats_raw['positive_votes'] / user_stats_raw['total_votes']

cluster_profile = (
    user_stats_raw.groupby('cluster')
    .agg(
        n_users=('raterParticipantId', 'nunique'),
        median_votes=('total_votes', 'median'),
        mean_votes=('total_votes', 'mean'),
        p25_votes=('total_votes', lambda x: x.quantile(0.25)),
        p75_votes=('total_votes', lambda x: x.quantile(0.75)),
        median_positive_rate=('positive_rate', 'median'),
        mean_positive_rate=('positive_rate', 'mean'),
        p25_positive_rate=('positive_rate', lambda x: x.quantile(0.25)),
        p75_positive_rate=('positive_rate', lambda x: x.quantile(0.75)),
    )
    .reset_index()
)
print('Cluster-level user profile summary:')
display(cluster_profile)

cluster_ids = sorted(user_stats_raw['cluster'].dropna().astype(int).unique())
colors = ['#D55E00', '#0072B2', '#009E73', '#CC79A7', '#E69F00']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for cluster_id in cluster_ids:
    data = user_stats_raw[user_stats_raw['cluster'] == cluster_id]['total_votes']
    axes[0].hist(data, bins=60, alpha=0.45, color=colors[cluster_id % len(colors)], label=f'Cluster {cluster_id}', density=True)
axes[0].set_xscale('log')
axes[0].set_title('Vote Count Distribution by Cluster\n(log scale)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Total votes per user (log scale)')
axes[0].set_ylabel('Density')
axes[0].legend(frameon=False)

box_data = [user_stats_raw[user_stats_raw['cluster'] == cluster_id]['positive_rate'].values for cluster_id in cluster_ids]
bp = axes[1].boxplot(
    box_data,
    tick_labels=[f'Cluster {cluster_id}' for cluster_id in cluster_ids],
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2),
    boxprops=dict(linewidth=0.8),
    whiskerprops=dict(linewidth=0.8),
    capprops=dict(linewidth=0.8),
    flierprops=dict(marker='o', markersize=2.5, alpha=0.3),
)
for patch, cluster_id in zip(bp['boxes'], cluster_ids):
    patch.set_facecolor(colors[cluster_id % len(colors)])
    patch.set_alpha(0.7)
axes[1].set_title('Positive Vote Rate Distribution by Cluster', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Positive vote rate')
axes[1].set_ylim(0, 1)
axes[1].axhline(0.5, color='grey', linestyle='--', linewidth=0.9, alpha=0.5)

plt.tight_layout()
plt.show()


## Persist Topic Artifacts

In [ ]:
save_table(topic_notes, PROCESSED_DIR / 'topic_notes.parquet')
save_table(topic_cluster_stats, PROCESSED_DIR / 'topic_cluster_stats.parquet')
save_table(exemplars, PROCESSED_DIR / 'topic_exemplars.parquet')
save_table(topic_salience, PROCESSED_DIR / 'topic_salience.parquet')
save_table(topic_salience_pivot, PROCESSED_DIR / 'topic_salience_pivot.parquet')
save_table(topic_rescue_stats, PROCESSED_DIR / 'topic_rescue_stats.parquet')
save_table(topic_strategy_summary, PROCESSED_DIR / 'topic_strategy_summary.parquet')
save_table(topic_strategy_pivot, PROCESSED_DIR / 'topic_strategy_pivot.parquet')
save_table(topic_selection_overlap, PROCESSED_DIR / 'topic_selection_overlap.parquet')

## Handoff to the LLM Phase

At this point you have a clean topic-enriched dataset that is also strategy-aware. For context classification, you can now join:
- note text
- cluster approvals
- bridge score
- disagreement features
- BERTopic labels or exemplar notes
- which selection rule surfaced the note and whether it remained `Helpful` or `NMR`

That gives the next LLM stage a much cleaner and more interpretable substrate than the original single notebook.

## Parent & Super-Parent Topic Layer

Fine-grained topics are now computed above. This section fits a coarser parent-topic layer on the **same corpus-specific subset** (`scores_for_topics`), then reduces parent topics to a compact super-parent layer for narrative use.

In [ ]:
# Model configuration. Change target_parent_topics to 15, 50, "auto", or None for quick variants.
embedding_model_name = 'all-MiniLM-L6-v2'
umap_neighbors = 15
umap_components = 5
hdbscan_min_cluster_size = 50
hdbscan_min_samples = 10
target_parent_topics = 30
random_state = 42

# Output and rerun safety.
# If all topic_parent_* outputs already exist, the notebook can skip the expensive BERTopic fit
# and use them for review/plots. To regenerate with new params, set this to False and
# set ALLOW_OVERWRITE_PARENT_OUTPUTS = False.
USE_EXISTING_PARENT_OUTPUTS_IF_AVAILABLE = True
PARENT_OUTPUT_PREFIX = 'topic_parent'
ALLOW_OVERWRITE_PARENT_OUTPUTS = False
SKIP_EXISTING_PARENT_OUTPUTS = True
CREATE_REASSIGNED_PARENT_NOTES = True
reassignment_embedding_model_name = embedding_model_name
target_super_parent_topics = 10
super_parent_linkage = 'ward'
candidate_super_parent_topics = [6, 8, 10, 12]
AUTO_SELECT_SUPER_PARENT_TOPIC_COUNT = True
MAX_SUPER_PARENT_SHARE = 0.35
MAX_PARENTS_PER_SUPER_PARENT = 7
MIN_MEAN_WITHIN_SUPER_PARENT_SIMILARITY = 0.18
LOW_REASSIGNMENT_SIMILARITY_THRESHOLD = 0.35

# Topic-representation cleanup. BERTopic is fit normally first; these terms are
# removed only from the c-TF-IDF/vectorizer representation before parent/super-parent reduction.
REMOVE_META_TERMS_FROM_TOPIC_LABELS = True
META_TOPIC_STOPWORDS = [
    'note', 'notes', 'community', 'birdwatch', 'nnn', 'cn',
    'comment', 'comments', 'section', 'needed', 'need',
    'helpful', 'contributors', 'contributor', 'rating', 'rated',
]
META_TERM_DETECTION_PATTERNS = [
    'community', 'note', 'notes', 'birdwatch', 'nnn', 'comment',
    'comments', 'section', 'contributor', 'contributors', 'rating', 'rated',
]

# Topic-level meta governance detection. These patterns flag whole topics whose
# summaries are mostly about note governance/no-note-needed disputes.
META_TOPIC_SUMMARY_PATTERNS = [
    r'\bnnn\b',
    r'no note needed',
    r'note not needed',
    r'no community note',
    r'not require(?:s|d)? (?:a )?(?:community )?note',
    r'keep it to (?:the )?comments',
    r'comment section',
    r'abus(?:e|ing) (?:community )?notes?',
    r'community notes?',
    r'\bcn\b',
]
META_DOMINANT_TOPIC_SHARE_THRESHOLD = 0.35
EXCLUDE_META_DOMINANT_TOPICS_FROM_REVIEW_PLOTS = True
EXCLUDE_META_DOMINANT_TOPICS_FROM_SUPER_PARENT = True

PARENT_OUTPUTS = {
    'notes': PROCESSED_DIR / 'topic_parent_notes.parquet',
    'notes_reassigned': PROCESSED_DIR / 'topic_parent_notes_reassigned.parquet',
    'cluster_stats': PROCESSED_DIR / 'topic_parent_cluster_stats.parquet',
    'exemplars': PROCESSED_DIR / 'topic_parent_exemplars.parquet',
    'crosswalk': PROCESSED_DIR / 'topic_parent_crosswalk.parquet',
    'diagnostics': PROCESSED_DIR / 'topic_parent_diagnostics.parquet',
}

PARENT_AUDIT_OUTPUTS = {
    'initial_topics': PROCESSED_DIR / 'topic_parent_initial_topics.parquet',
    'initial_topic_terms': PROCESSED_DIR / 'topic_parent_initial_topic_terms.parquet',
    'meta_terms': PROCESSED_DIR / 'topic_parent_meta_terms.parquet',
    'meta_topic_diagnostics': PROCESSED_DIR / 'topic_parent_meta_topic_diagnostics.parquet',
    'reassignment_sensitivity': PROCESSED_DIR / 'topic_parent_reassignment_sensitivity.parquet',
}

SUPER_PARENT_AUDIT_OUTPUTS = {
    'candidate_quality': PROCESSED_DIR / 'topic_super_parent_candidate_quality.parquet',
    'candidate_crosswalk': PROCESSED_DIR / 'topic_super_parent_candidate_crosswalk.parquet',
}

SUPER_PARENT_OUTPUTS = {
    'notes': PROCESSED_DIR / 'topic_super_parent_notes.parquet',
    'cluster_stats': PROCESSED_DIR / 'topic_super_parent_cluster_stats.parquet',
    'crosswalk': PROCESSED_DIR / 'topic_super_parent_crosswalk.parquet',
    'diagnostics': PROCESSED_DIR / 'topic_super_parent_diagnostics.parquet',
}

display(pd.DataFrame([{
    'embedding_model_name': embedding_model_name,
    'umap_neighbors': umap_neighbors,
    'umap_components': umap_components,
    'hdbscan_min_cluster_size': hdbscan_min_cluster_size,
    'hdbscan_min_samples': hdbscan_min_samples,
    'target_parent_topics': target_parent_topics,
    'random_state': random_state,
    'use_existing_parent_outputs_if_available': USE_EXISTING_PARENT_OUTPUTS_IF_AVAILABLE,
    'allow_overwrite': ALLOW_OVERWRITE_PARENT_OUTPUTS,
    'skip_existing_parent_outputs': SKIP_EXISTING_PARENT_OUTPUTS,
    'create_reassigned_parent_notes': CREATE_REASSIGNED_PARENT_NOTES,
    'reassignment_embedding_model_name': reassignment_embedding_model_name,
    'target_super_parent_topics': target_super_parent_topics,
    'candidate_super_parent_topics': ', '.join(map(str, candidate_super_parent_topics)),
    'auto_select_super_parent_topic_count': AUTO_SELECT_SUPER_PARENT_TOPIC_COUNT,
    'super_parent_linkage': super_parent_linkage,
    'remove_meta_terms_from_topic_labels': REMOVE_META_TERMS_FROM_TOPIC_LABELS,
    'meta_topic_stopwords': ', '.join(META_TOPIC_STOPWORDS),
    'meta_dominant_topic_share_threshold': META_DOMINANT_TOPIC_SHARE_THRESHOLD,
    'exclude_meta_dominant_topics_from_review_plots': EXCLUDE_META_DOMINANT_TOPICS_FROM_REVIEW_PLOTS,
    'exclude_meta_dominant_topics_from_super_parent': EXCLUDE_META_DOMINANT_TOPICS_FROM_SUPER_PARENT,
}]))

## Existing Parent Outputs

If the parent-topic artifacts already exist, this notebook defaults to review mode: it loads those artifacts, skips the expensive BERTopic fit, and still renders the diagnostics/plots below. Set `USE_EXISTING_PARENT_OUTPUTS_IF_AVAILABLE = False` and `ALLOW_OVERWRITE_PARENT_OUTPUTS = True` only when you intentionally want to regenerate the parent layer.

In [ ]:
BASE_PARENT_OUTPUT_KEYS = ['notes', 'cluster_stats', 'exemplars', 'crosswalk', 'diagnostics']
existing_parent_outputs = {key: path for key, path in PARENT_OUTPUTS.items() if path.exists()}
missing_base_parent_outputs = {key: PARENT_OUTPUTS[key] for key in BASE_PARENT_OUTPUT_KEYS if not PARENT_OUTPUTS[key].exists()}
all_base_parent_outputs_exist = not missing_base_parent_outputs
USE_EXISTING_PARENT_OUTPUTS = (
    USE_EXISTING_PARENT_OUTPUTS_IF_AVAILABLE
    and all_base_parent_outputs_exist
    and not ALLOW_OVERWRITE_PARENT_OUTPUTS
)

parent_notes = None
parent_cluster_stats = None
parent_exemplars = None
topic_parent_crosswalk = None
topic_parent_diagnostics = None
topic_parent_notes_reassigned = None
parent_notes_raw = None
parent_topic_model = None
parent_embeddings = None
initial_topic_count = None
parent_initial_topics = pd.DataFrame()
parent_initial_topic_terms = pd.DataFrame()
topic_parent_meta_terms = pd.DataFrame()
topic_parent_meta_topic_diagnostics = pd.DataFrame()
topic_parent_reassignment_sensitivity = pd.DataFrame()

if USE_EXISTING_PARENT_OUTPUTS:
    parent_notes = load_table(PARENT_OUTPUTS['notes'])
    parent_cluster_stats = load_table(PARENT_OUTPUTS['cluster_stats'])
    parent_exemplars = load_table(PARENT_OUTPUTS['exemplars'])
    topic_parent_crosswalk = load_table(PARENT_OUTPUTS['crosswalk'])
    topic_parent_diagnostics = load_table(PARENT_OUTPUTS['diagnostics'])
    if PARENT_OUTPUTS['notes_reassigned'].exists():
        topic_parent_notes_reassigned = load_table(PARENT_OUTPUTS['notes_reassigned'])
    parent_notes_raw = parent_notes.copy()
    if PARENT_AUDIT_OUTPUTS['initial_topics'].exists():
        parent_initial_topics = load_table(PARENT_AUDIT_OUTPUTS['initial_topics'])
    if PARENT_AUDIT_OUTPUTS['initial_topic_terms'].exists():
        parent_initial_topic_terms = load_table(PARENT_AUDIT_OUTPUTS['initial_topic_terms'])
    if PARENT_AUDIT_OUTPUTS['meta_terms'].exists():
        topic_parent_meta_terms = load_table(PARENT_AUDIT_OUTPUTS['meta_terms'])
    if PARENT_AUDIT_OUTPUTS['meta_topic_diagnostics'].exists():
        topic_parent_meta_topic_diagnostics = load_table(PARENT_AUDIT_OUTPUTS['meta_topic_diagnostics'])
    if PARENT_AUDIT_OUTPUTS['reassignment_sensitivity'].exists():
        topic_parent_reassignment_sensitivity = load_table(PARENT_AUDIT_OUTPUTS['reassignment_sensitivity'])
    if not topic_parent_diagnostics.empty and 'initial_topics_non_outlier' in topic_parent_diagnostics.columns:
        initial_topic_count = int(topic_parent_diagnostics.loc[0, 'initial_topics_non_outlier'])
    elif 'initial_topic' in parent_notes.columns:
        initial_topic_count = parent_notes.loc[parent_notes['initial_topic'].ne(-1), 'initial_topic'].nunique()
    else:
        initial_topic_count = pd.NA
    print('Existing base topic_parent_* outputs found; using review mode and skipping BERTopic fit.')
    if topic_parent_notes_reassigned is not None:
        print('Existing topic_parent_notes_reassigned.parquet found; it will be loaded and reviewed.')
    else:
        print('topic_parent_notes_reassigned.parquet not found; it will be built from strict parent notes.')
else:
    if existing_parent_outputs:
        print('Partial parent outputs found; BERTopic will run and outputs will be written/skipped according to persist settings.')
        display(pd.DataFrame({'artifact': list(existing_parent_outputs), 'path': [str(path) for path in existing_parent_outputs.values()]}))
    else:
        print('No existing topic_parent_* outputs found; BERTopic will run.')
    if missing_base_parent_outputs:
        print('Missing required base parent outputs:')
        display(pd.DataFrame({'artifact': list(missing_base_parent_outputs), 'path': [str(path) for path in missing_base_parent_outputs.values()]}))

In [ ]:
# Parent topic input = same prepared corpus-specific subset as fine-grained topics above.
# `prepare_topic_frame` is where canonical topic metrics such as `abs_gap`
# are produced, so parent topics must use that prepared frame too.
parent_topic_input = topic_input.copy()
docs = parent_topic_input['summary'].fillna('').tolist()
print(
    f'Parent topic input: {len(parent_topic_input):,} prepared notes '
    '(same corpus subset as fine topics)'
)

# Load fine topic notes for crosswalk (already computed above).
fine_topic_notes = topic_notes.copy()

## Fit Coarse BERTopic

The clustering step is deliberately more conservative than the fine-grained workflow. HDBSCAN starts with a larger minimum cluster size, and BERTopic then reduces topics to the configured parent-topic target.


In [ ]:
def _make_topic_vectorizer(extra_stopwords: list[str] | None = None):
    from sklearn.feature_extraction import text as sk_text
    from sklearn.feature_extraction.text import CountVectorizer

    extra_stopwords = extra_stopwords or []
    stopwords = set(sk_text.ENGLISH_STOP_WORDS).union(_EXTRA_STOP_WORDS)
    # CountVectorizer stop_words are token-level; keep only single alphabetic tokens.
    stopwords.update(
        str(term).lower().strip()
        for term in extra_stopwords
        if isinstance(term, str) and term.strip().isalpha() and ' ' not in term.strip()
    )
    return CountVectorizer(
        stop_words=sorted(stopwords),
        min_df=2,
        ngram_range=(1, 2),
        token_pattern=r'(?u)\b[A-Za-z][A-Za-z]{2,}\b',
    )


def extract_topic_terms(topic_model, *, stage: str, top_n: int = 30) -> pd.DataFrame:
    rows = []
    for topic_id in sorted(t for t in topic_model.get_topics().keys() if t != -1):
        for rank, item in enumerate(topic_model.get_topic(topic_id)[:top_n], start=1):
            term, weight = item
            rows.append({
                'stage': stage,
                'topic': int(topic_id),
                'rank': rank,
                'term': str(term),
                'weight': float(weight),
            })
    return pd.DataFrame(rows)


def detect_meta_terms(initial_terms: pd.DataFrame) -> pd.DataFrame:
    if initial_terms.empty:
        return pd.DataFrame(columns=['term', 'source', 'matched_pattern', 'topic_count', 'max_weight'])
    patterns = [p.lower() for p in META_TERM_DETECTION_PATTERNS]
    seed_terms = {t.lower() for t in META_TOPIC_STOPWORDS}
    rows = []
    for term, grp in initial_terms.groupby(initial_terms['term'].str.lower()):
        term_tokens = set(term.split())
        matched = sorted({p for p in patterns if p in term_tokens or p == term})
        if term in seed_terms or term_tokens.intersection(seed_terms) or matched:
            rows.append({
                'term': term,
                'source': 'seed_or_detected',
                'matched_pattern': ', '.join(matched) if matched else 'seed_stopword',
                'topic_count': int(grp['topic'].nunique()),
                'max_weight': float(grp['weight'].max()),
            })
    meta_df = pd.DataFrame(rows)
    seed_only = pd.DataFrame([
        {'term': term, 'source': 'seed_stopword', 'matched_pattern': 'seed_stopword', 'topic_count': 0, 'max_weight': np.nan}
        for term in sorted(seed_terms)
        if meta_df.empty or term not in set(meta_df['term'])
    ])
    return pd.concat([meta_df, seed_only], ignore_index=True).sort_values(['source', 'topic_count', 'max_weight'], ascending=[True, False, False])


def fit_parent_topic_model(
    topic_df: pd.DataFrame,
    embedding_model_name: str,
    umap_neighbors: int,
    umap_components: int,
    hdbscan_min_cluster_size: int,
    hdbscan_min_samples: int,
    target_parent_topics,
    random_state: int,
):
    from bertopic import BERTopic
    from hdbscan import HDBSCAN
    from sentence_transformers import SentenceTransformer
    from umap import UMAP

    texts = topic_df['summary'].astype(str).tolist()
    embedder = SentenceTransformer(embedding_model_name)
    embeddings = embedder.encode(texts, show_progress_bar=True)

    umap_model = UMAP(
        n_neighbors=umap_neighbors,
        n_components=umap_components,
        min_dist=0.0,
        metric='cosine',
        random_state=random_state,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=hdbscan_min_cluster_size,
        min_samples=hdbscan_min_samples,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True,
    )

    # Stage 1: fit normal BERTopic first, with the project baseline stopwords only.
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=_make_topic_vectorizer(),
        calculate_probabilities=False,
        verbose=True,
    )
    initial_topics, probs = topic_model.fit_transform(texts, embeddings)
    initial_topic_count = len({topic for topic in initial_topics if topic != -1})
    initial_topic_info = topic_model.get_topic_info().copy()
    initial_topic_info['stage'] = 'initial_bertopic_before_meta_cleanup'
    initial_topic_terms = extract_topic_terms(
        topic_model,
        stage='initial_bertopic_before_meta_cleanup',
        top_n=30,
    )
    meta_terms = detect_meta_terms(initial_topic_terms)

    # Stage 2: remove meta vocabulary from topic representation only, then reduce.
    if REMOVE_META_TERMS_FROM_TOPIC_LABELS:
        fine_meta = globals().get('fine_topic_meta_terms', pd.DataFrame())
        fine_meta_terms = (
            set(fine_meta['term'].astype(str).str.lower())
            if isinstance(fine_meta, pd.DataFrame) and not fine_meta.empty and 'term' in fine_meta.columns
            else set()
        )
        if fine_meta_terms:
            meta_terms = pd.concat([meta_terms, fine_meta.assign(used_for_cleanup=True)], ignore_index=True)
        cleanup_terms = sorted(set(META_TOPIC_STOPWORDS).union(meta_terms['term'].astype(str).str.lower()).union(fine_meta_terms))
        topic_model.update_topics(texts, vectorizer_model=_make_topic_vectorizer(cleanup_terms))
        cleaned_topic_terms = extract_topic_terms(
            topic_model,
            stage='after_meta_term_cleanup_before_reduction',
            top_n=30,
        )
        initial_topic_terms = pd.concat([initial_topic_terms, cleaned_topic_terms], ignore_index=True)
    else:
        cleanup_terms = []

    if target_parent_topics not in (None, 0, 'none', 'None'):
        topic_model.reduce_topics(texts, nr_topics=target_parent_topics)

    enriched = topic_df.copy()
    enriched['initial_topic'] = initial_topics
    enriched['topic'] = topic_model.topics_
    enriched['topic_probability'] = None

    meta_terms = meta_terms.copy()
    if 'used_for_cleanup' in meta_terms.columns:
        meta_terms['used_for_cleanup'] = meta_terms['used_for_cleanup'].fillna(
            meta_terms['term'].astype(str).str.lower().isin(cleanup_terms)
        )
    else:
        meta_terms['used_for_cleanup'] = meta_terms['term'].astype(str).str.lower().isin(cleanup_terms)

    return topic_model, enriched, embeddings, initial_topic_count, initial_topic_info, initial_topic_terms, meta_terms


In [ ]:
if USE_EXISTING_PARENT_OUTPUTS:
    print('Skipping BERTopic fit because existing parent outputs are loaded.')
    print(f'Initial non-outlier topics before reduction: {initial_topic_count}')
    print(f'Final topics including outlier: {parent_notes_raw["topic"].nunique():,}')
    print(f'Final non-outlier parent topics: {parent_notes_raw.loc[parent_notes_raw["topic"].ne(-1), "topic"].nunique():,}')
else:
    parent_topic_model, parent_notes_raw, parent_embeddings, initial_topic_count, parent_initial_topics, parent_initial_topic_terms, topic_parent_meta_terms = fit_parent_topic_model(
        parent_topic_input,
        embedding_model_name=embedding_model_name,
        umap_neighbors=umap_neighbors,
        umap_components=umap_components,
        hdbscan_min_cluster_size=hdbscan_min_cluster_size,
        hdbscan_min_samples=hdbscan_min_samples,
        target_parent_topics=target_parent_topics,
        random_state=random_state,
    )

    print(f'Initial non-outlier topics before reduction: {initial_topic_count:,}')
    print(f'Final topics including outlier: {parent_notes_raw["topic"].nunique():,}')
    print(f'Final non-outlier parent topics: {parent_notes_raw.loc[parent_notes_raw["topic"].ne(-1), "topic"].nunique():,}')

## Initial Topic Audit & Meta-Term Cleanup

The notebook first fits normal BERTopic and records its unreduced topic vocabulary. Meta tokens detected from that initial vocabulary are removed only from the topic representation before the parent and super-parent reductions continue.

In [ ]:
if not parent_initial_topics.empty:
    print('Initial BERTopic topic list before meta-term cleanup')
    display(parent_initial_topics.head(50))
else:
    print('No initial topic audit table loaded or generated in this run.')

if not topic_parent_meta_terms.empty:
    print('Meta terms detected from initial topic vocabulary')
    display(topic_parent_meta_terms.head(80))
else:
    print('No meta term diagnostics loaded or generated in this run.')

if not parent_initial_topic_terms.empty:
    term_counts = (
        parent_initial_topic_terms
        .groupby(['stage', 'topic'], dropna=False)
        .head(12)
        .reset_index(drop=True)
    )
    display(term_counts.head(120))


## Meta-Term Filtering Audit Notes

This workflow intentionally does **not** remove notes from the corpus. It first fits the normal unreduced BERTopic model, records the initial topic vocabulary, then removes meta/process vocabulary only from the BERTopic c-TF-IDF representation before the parent-topic and super-parent-topic reductions. This keeps the embedding clusters intact while preventing Community Notes process terms from dominating topic labels.

Terms filtered from topic representations in the current run:

- Seed/process terms: `note`, `notes`, `community`, `community notes`, `community note`, `birdwatch`, `nnn`, `cn`, `comment`, `comments`, `section`, `helpful`, `contributor`, `contributors`, `rating`, `rated`, `need`, `needed`.
- Terms detected from the initial BERTopic vocabulary: `abusing community`, `comments nnn`, `joke nnn`, `nnn clearly`, `nnn obviously`, `opinion nnn`.
- Terms additionally detected from the existing fine-topic labels: `community`, `comments`, `notes`.

Quality check after filtering:

- Parent-topic labels contain zero matches for `Community`, `Notes`, `Note`, `NNN`, `Birdwatch`, `comment(s)`, `section`, `contributors`, `rating`, or `rated`.
- Super-parent labels also contain zero matches for those meta/process terms.
- The previous label `Notes Community Community Notes` no longer appears. Its residual semantic cluster is now labelled `Opinion Abusing Stop Stop`.

Interpretation caveat for the paper:

- `Opinion Abusing Stop Stop` is still a **meta-governance / no-note-needed dispute** topic. The filtering removed the process vocabulary from the label, but the underlying notes remain semantically similar because many summaries are genuinely about “NNN,” “keep it to comments,” “opinion,” or “do not abuse Community Notes.” If the analysis needs only substantive tweet-content topics, this topic should be flagged/excluded at the analysis stage rather than handled by vocabulary cleanup alone.
- `Image Photo Altered Video` should **not** be treated as a meta-topic by default. Its summaries are mostly about media authenticity, edited images, generated images, screenshots, and altered videos. It is a substantive content-verification topic, even though a small number of its notes mention Community Notes or NNN incidentally.


Current operational rule after the latest run:

- Meta-term filtering cleans labels/vocabulary.
- Topic-level meta-governance detection separately flags parent topics where at least 35% of summaries match no-note-needed / Community Notes governance patterns.
- The current run flags `Opinion Abusing Stop Stop` as meta-dominant (`597 / 705 = 84.7%` meta-governance notes).
- Meta-dominant parent topics are retained in `topic_parent_*` artifacts for auditability, but excluded from review plots and from the `topic_super_parent_*` narrative layer.


## Quality Evaluation Roadmap

This notebook now treats the final topic model as a layered pipeline rather than a single fixed clustering. First, it fits BERTopic and cleans process/meta vocabulary from topic labels. Second, it flags meta-governance topics whose summaries are mostly about no-note-needed or Community Notes process disputes. Third, it reassigns HDBSCAN outliers to the nearest parent centroid but reports low-similarity reassignments as a sensitivity check. Fourth, it evaluates candidate super-parent counts (`6`, `8`, `10`, `12`) and selects the smallest candidate that passes concentration and conceptual-coherence gates. The parent reassigned layer remains the detailed analytic layer; the super-parent layer is the paper-facing narrative layer.

## Attach Labels & Build Tables

In [ ]:
if USE_EXISTING_PARENT_OUTPUTS:
    print('Using loaded parent note/stat/exemplar tables.')
else:
    parent_notes = attach_topic_names(parent_topic_model, parent_notes_raw)
    parent_notes = add_topic_display_labels(parent_notes)

    parent_cluster_stats = build_topic_cluster_stats(parent_notes)
    parent_cluster_stats = add_topic_display_labels(parent_cluster_stats)

    def build_parent_exemplars(parent_notes: pd.DataFrame, parent_stats: pd.DataFrame, top_k_exemplars: int = 5) -> pd.DataFrame:
        parent_topics = parent_stats['topic'].tolist()
        exemplars = (
            parent_notes[parent_notes['topic'].isin(parent_topics)]
            .sort_values(['topic', 'abs_gap', 'total_votes'], ascending=[True, False, False])
            .groupby('topic')
            .head(top_k_exemplars)
            .reset_index(drop=True)
        )
        return exemplars

    parent_exemplars = build_parent_exemplars(parent_notes, parent_cluster_stats, top_k_exemplars=5)

display(parent_notes.head())
display(parent_cluster_stats.head(20))
display(parent_exemplars.head(20))


## Meta-Dominant Topic Flags

Meta-term cleanup removes process words from topic labels. This section separately flags whole topics whose summaries are dominated by no-note-needed / Community Notes governance language. These topics are kept in the parent artifacts for auditability, but can be excluded from paper-facing review plots and super-parent narrative reduction.

In [ ]:
import re


def add_meta_dominant_topic_flags(
    notes: pd.DataFrame,
    stats: pd.DataFrame,
    patterns: list[str],
    threshold: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if notes is None or notes.empty:
        return notes, stats, pd.DataFrame()
    regex = re.compile('|'.join(f'(?:{pattern})' for pattern in patterns), flags=re.IGNORECASE)
    flagged_notes = notes.copy()
    flagged_notes['is_meta_governance_note'] = flagged_notes['summary'].astype(str).str.contains(regex, na=False)
    meta_summary = (
        flagged_notes.groupby('topic', dropna=False)
        .agg(
            notes=('noteId', 'nunique'),
            meta_governance_notes=('is_meta_governance_note', 'sum'),
        )
        .reset_index()
    )
    meta_summary['meta_governance_note_share'] = (
        meta_summary['meta_governance_notes'] / meta_summary['notes'].clip(lower=1)
    )
    meta_summary['is_meta_dominant_topic'] = (
        meta_summary['topic'].ne(-1)
        & meta_summary['meta_governance_note_share'].ge(threshold)
    )
    flagged_notes = flagged_notes.merge(
        meta_summary[['topic', 'meta_governance_note_share', 'is_meta_dominant_topic']],
        on='topic',
        how='left',
    )
    flagged_stats = stats.merge(
        meta_summary[['topic', 'meta_governance_notes', 'meta_governance_note_share', 'is_meta_dominant_topic']],
        on='topic',
        how='left',
    )
    flagged_stats['is_meta_dominant_topic'] = flagged_stats['is_meta_dominant_topic'].fillna(False)
    return flagged_notes, flagged_stats, meta_summary


parent_notes, parent_cluster_stats, topic_parent_meta_topic_diagnostics = add_meta_dominant_topic_flags(
    parent_notes,
    parent_cluster_stats,
    META_TOPIC_SUMMARY_PATTERNS,
    META_DOMINANT_TOPIC_SHARE_THRESHOLD,
)

if not parent_exemplars.empty and 'topic' in parent_exemplars.columns:
    parent_exemplars = parent_exemplars.merge(
        topic_parent_meta_topic_diagnostics[['topic', 'meta_governance_note_share', 'is_meta_dominant_topic']],
        on='topic',
        how='left',
    )

print('Meta-dominant topic diagnostics')
display(
    topic_parent_meta_topic_diagnostics
    .sort_values('meta_governance_note_share', ascending=False)
    .head(20)
)
print('Flagged meta-dominant parent topics')
display(
    parent_cluster_stats[parent_cluster_stats['is_meta_dominant_topic']]
    [['topic', 'topic_label', 'notes', 'meta_governance_notes', 'meta_governance_note_share']]
    .sort_values('meta_governance_note_share', ascending=False)
)


## Fine Topic Crosswalk

If the existing fine-grained 246-topic output is present, this crosswalk shows where those topics land in the new model-generated parent-topic layer.


In [ ]:
def build_topic_parent_crosswalk(fine_topic_notes: pd.DataFrame | None, parent_notes: pd.DataFrame) -> pd.DataFrame:
    columns = [
        'fine_topic',
        'fine_topic_label',
        'parent_topic',
        'parent_topic_label',
        'notes',
        'share_within_fine_topic',
        'is_dominant_parent',
    ]
    if fine_topic_notes is None:
        return pd.DataFrame(columns=columns)

    fine_cols = [col for col in ['noteId', 'topic', 'topic_label', 'topic_display_label'] if col in fine_topic_notes.columns]
    parent_cols = [col for col in ['noteId', 'topic', 'topic_label', 'topic_display_label'] if col in parent_notes.columns]

    fine = fine_topic_notes[fine_cols].drop_duplicates('noteId').copy()
    parent = parent_notes[parent_cols].drop_duplicates('noteId').copy()
    fine['noteId'] = fine['noteId'].astype(str)
    parent['noteId'] = parent['noteId'].astype(str)

    fine_label_col = 'topic_display_label' if 'topic_display_label' in fine.columns else 'topic_label'
    parent_label_col = 'topic_display_label' if 'topic_display_label' in parent.columns else 'topic_label'
    fine = fine.rename(columns={'topic': 'fine_topic', fine_label_col: 'fine_topic_label'})
    parent = parent.rename(columns={'topic': 'parent_topic', parent_label_col: 'parent_topic_label'})

    merged = fine[['noteId', 'fine_topic', 'fine_topic_label']].merge(
        parent[['noteId', 'parent_topic', 'parent_topic_label']],
        on='noteId',
        how='inner',
    )
    if merged.empty:
        return pd.DataFrame(columns=columns)

    crosswalk = (
        merged.groupby(['fine_topic', 'fine_topic_label', 'parent_topic', 'parent_topic_label'])
        .agg(notes=('noteId', 'nunique'))
        .reset_index()
    )
    fine_totals = crosswalk.groupby('fine_topic')['notes'].transform('sum')
    max_within_fine = crosswalk.groupby('fine_topic')['notes'].transform('max')
    crosswalk['share_within_fine_topic'] = crosswalk['notes'] / fine_totals
    crosswalk['is_dominant_parent'] = crosswalk['notes'].eq(max_within_fine)
    return crosswalk.sort_values(['fine_topic', 'is_dominant_parent', 'notes'], ascending=[True, False, False])

if USE_EXISTING_PARENT_OUTPUTS and topic_parent_crosswalk is not None:
    print('Using loaded topic_parent_crosswalk.')
else:
    topic_parent_crosswalk = build_topic_parent_crosswalk(fine_topic_notes, parent_notes)

print(f'Crosswalk rows: {len(topic_parent_crosswalk):,}')
display(topic_parent_crosswalk.head(30))

## Diagnostics

In [ ]:
topic_sizes = parent_notes.groupby('topic')['noteId'].nunique().sort_values(ascending=False)
non_outlier_sizes = topic_sizes[topic_sizes.index != -1]
outlier_notes = int((parent_notes['topic'] == -1).sum())

if USE_EXISTING_PARENT_OUTPUTS and topic_parent_diagnostics is not None and not topic_parent_diagnostics.empty:
    print('Using loaded topic_parent_diagnostics.')
else:
    topic_parent_diagnostics = pd.DataFrame([{
        'pipeline': 'canonical_200k_representative',
        'input_rows': len(parent_topic_input),
        'unique_notes': parent_topic_input['noteId'].nunique(),
        'initial_topics_non_outlier': initial_topic_count,
        'final_topics_including_outlier': parent_notes['topic'].nunique(),
        'final_parent_topics_non_outlier': parent_notes.loc[parent_notes['topic'].ne(-1), 'topic'].nunique(),
        'outlier_notes': outlier_notes,
        'outlier_share': outlier_notes / max(len(parent_notes), 1),
        'median_parent_topic_size': float(non_outlier_sizes.median()) if not non_outlier_sizes.empty else np.nan,
        'max_parent_topic_size': int(non_outlier_sizes.max()) if not non_outlier_sizes.empty else 0,
        'embedding_model_name': embedding_model_name,
        'umap_neighbors': umap_neighbors,
        'umap_components': umap_components,
        'hdbscan_min_cluster_size': hdbscan_min_cluster_size,
        'hdbscan_min_samples': hdbscan_min_samples,
        'target_parent_topics': str(target_parent_topics),
        'random_state': random_state,
        'remove_meta_terms_from_topic_labels': REMOVE_META_TERMS_FROM_TOPIC_LABELS,
        'meta_terms_used_for_cleanup': int(topic_parent_meta_terms['used_for_cleanup'].sum()) if not topic_parent_meta_terms.empty and 'used_for_cleanup' in topic_parent_meta_terms.columns else 0,
    }])

display(topic_parent_diagnostics)
display(topic_sizes.rename_axis('topic').reset_index(name='notes').head(40))

## Outlier Reassignment

The strict parent layer keeps BERTopic/HDBSCAN outliers as `topic = -1`. For coverage-oriented analysis, this section creates a second note-level table where only those outlier notes are assigned to the nearest non-outlier parent topic centroid in SentenceTransformer embedding space. The original strict assignment is preserved in `original_parent_*` columns.

In [ ]:
def _l2_normalize(matrix: np.ndarray) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=np.float32)
    matrix = np.nan_to_num(matrix, nan=0.0, posinf=0.0, neginf=0.0)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return matrix / norms


def get_parent_embeddings_for_reassignment(parent_notes: pd.DataFrame, existing_embeddings=None) -> np.ndarray:
    if existing_embeddings is not None:
        return _l2_normalize(np.asarray(existing_embeddings, dtype=np.float32))

    from sentence_transformers import SentenceTransformer

    texts = parent_notes['summary'].astype(str).tolist()
    embedder = SentenceTransformer(reassignment_embedding_model_name)
    embeddings = embedder.encode(texts, show_progress_bar=True)
    return _l2_normalize(embeddings)


def build_reassigned_parent_notes(parent_notes: pd.DataFrame, embeddings: np.ndarray) -> pd.DataFrame:
    notes = parent_notes.reset_index(drop=True).copy()
    embeddings = _l2_normalize(embeddings)
    if len(notes) != len(embeddings):
        raise ValueError(f'Embedding row count mismatch: notes={len(notes):,}, embeddings={len(embeddings):,}')

    notes['original_parent_topic'] = notes['topic']
    if 'Name' in notes.columns:
        notes['original_parent_Name'] = notes['Name']
    if 'topic_label' in notes.columns:
        notes['original_parent_topic_label'] = notes['topic_label']
    if 'topic_display_label' in notes.columns:
        notes['original_parent_topic_display_label'] = notes['topic_display_label']
    notes['reassigned_from_outlier'] = notes['topic'].eq(-1)
    notes['parent_reassignment_similarity'] = np.nan
    notes['parent_reassignment_method'] = 'unchanged_non_outlier'

    outlier_mask = notes['topic'].eq(-1).to_numpy()
    non_outlier_mask = ~outlier_mask
    if not outlier_mask.any():
        return notes
    if not non_outlier_mask.any():
        raise ValueError('Cannot reassign parent outliers because there are no non-outlier parent topics.')

    parent_topics = sorted(notes.loc[non_outlier_mask, 'topic'].dropna().unique())
    centroids = []
    for topic in parent_topics:
        topic_positions = notes.index[notes['topic'].eq(topic)].to_numpy()
        centroid = embeddings[topic_positions].mean(axis=0, keepdims=True)
        centroids.append(_l2_normalize(centroid)[0])
    centroid_matrix = np.vstack(centroids)

    outlier_positions = np.flatnonzero(outlier_mask)
    similarities = embeddings[outlier_positions] @ centroid_matrix.T
    best_centroid_idx = similarities.argmax(axis=1)
    assigned_topics = np.asarray(parent_topics, dtype=notes['topic'].dtype)[best_centroid_idx]
    assigned_similarity = similarities[np.arange(len(outlier_positions)), best_centroid_idx]

    notes.loc[outlier_positions, 'topic'] = assigned_topics
    notes.loc[outlier_positions, 'parent_reassignment_similarity'] = assigned_similarity.astype(float)
    notes.loc[outlier_positions, 'parent_reassignment_method'] = f'nearest_parent_centroid_{reassignment_embedding_model_name}'
    if 'Topic' in notes.columns:
        notes.loc[:, 'Topic'] = notes['topic']

    label_columns = [col for col in ['Name', 'topic_label', 'topic_display_label'] if col in notes.columns]
    topic_meta = (
        parent_notes[parent_notes['topic'].ne(-1)]
        .loc[:, ['topic', *label_columns]]
        .drop_duplicates('topic')
    )
    for col in label_columns:
        label_map = topic_meta.set_index('topic')[col]
        notes.loc[:, col] = notes['topic'].map(label_map).fillna(notes[col])

    return notes


if not CREATE_REASSIGNED_PARENT_NOTES:
    topic_parent_notes_reassigned = pd.DataFrame()
    topic_parent_reassignment_diagnostics = pd.DataFrame([{'status': 'disabled'}])
elif topic_parent_notes_reassigned is not None and not topic_parent_notes_reassigned.empty and not ALLOW_OVERWRITE_PARENT_OUTPUTS:
    print('Using loaded topic_parent_notes_reassigned.parquet.')
    reassigned_outliers = int(topic_parent_notes_reassigned.get('reassigned_from_outlier', pd.Series(dtype=bool)).sum())
    topic_parent_reassignment_diagnostics = pd.DataFrame([{
        'status': 'loaded_existing',
        'strict_outlier_notes': int(parent_notes['topic'].eq(-1).sum()),
        'reassigned_outlier_notes': reassigned_outliers,
        'remaining_outlier_notes': int(topic_parent_notes_reassigned['topic'].eq(-1).sum()),
        'mean_reassignment_similarity': float(topic_parent_notes_reassigned.loc[
            topic_parent_notes_reassigned.get('reassigned_from_outlier', False),
            'parent_reassignment_similarity',
        ].mean()) if 'parent_reassignment_similarity' in topic_parent_notes_reassigned.columns else np.nan,
    }])
else:
    reassignment_embeddings = get_parent_embeddings_for_reassignment(parent_notes, parent_embeddings)
    topic_parent_notes_reassigned = build_reassigned_parent_notes(parent_notes, reassignment_embeddings)
    reassigned_mask = topic_parent_notes_reassigned['reassigned_from_outlier']
    topic_parent_reassignment_diagnostics = pd.DataFrame([{
        'status': 'built',
        'strict_outlier_notes': int(parent_notes['topic'].eq(-1).sum()),
        'reassigned_outlier_notes': int(reassigned_mask.sum()),
        'remaining_outlier_notes': int(topic_parent_notes_reassigned['topic'].eq(-1).sum()),
        'mean_reassignment_similarity': float(topic_parent_notes_reassigned.loc[reassigned_mask, 'parent_reassignment_similarity'].mean()),
        'median_reassignment_similarity': float(topic_parent_notes_reassigned.loc[reassigned_mask, 'parent_reassignment_similarity'].median()),
        'min_reassignment_similarity': float(topic_parent_notes_reassigned.loc[reassigned_mask, 'parent_reassignment_similarity'].min()),
        'reassignment_embedding_model_name': reassignment_embedding_model_name,
    }])

meta_flag_cols = [
    col for col in ['topic', 'meta_governance_note_share', 'is_meta_dominant_topic']
    if col in parent_cluster_stats.columns
]
if meta_flag_cols and topic_parent_notes_reassigned is not None and not topic_parent_notes_reassigned.empty:
    topic_parent_notes_reassigned = topic_parent_notes_reassigned.drop(
        columns=[col for col in ['meta_governance_note_share', 'is_meta_dominant_topic'] if col in topic_parent_notes_reassigned.columns],
        errors='ignore',
    ).merge(parent_cluster_stats[meta_flag_cols].drop_duplicates('topic'), on='topic', how='left')
    topic_parent_notes_reassigned['is_meta_dominant_topic'] = topic_parent_notes_reassigned['is_meta_dominant_topic'].fillna(False)

display(topic_parent_reassignment_diagnostics)
display(topic_parent_notes_reassigned.head())

## Outlier Reassignment Sensitivity

This diagnostic checks whether nearest-centroid reassignment creates low-confidence parent assignments. Low-similarity reassigned outliers are kept in the reassigned table, but their volume and topic concentration are reported for robustness checks.

In [ ]:
def build_reassignment_sensitivity(notes_reassigned: pd.DataFrame, threshold: float) -> pd.DataFrame:
    if notes_reassigned is None or notes_reassigned.empty or 'reassigned_from_outlier' not in notes_reassigned.columns:
        return pd.DataFrame()
    reassigned = notes_reassigned[notes_reassigned['reassigned_from_outlier']].copy()
    if reassigned.empty:
        return pd.DataFrame([{
            'scope': 'all_reassigned_outliers',
            'reassigned_outlier_notes': 0,
            'low_similarity_threshold': threshold,
            'low_similarity_notes': 0,
            'low_similarity_share': 0.0,
        }])
    summary = pd.DataFrame([{
        'scope': 'all_reassigned_outliers',
        'reassigned_outlier_notes': len(reassigned),
        'low_similarity_threshold': threshold,
        'mean_similarity': float(reassigned['parent_reassignment_similarity'].mean()),
        'median_similarity': float(reassigned['parent_reassignment_similarity'].median()),
        'p10_similarity': float(reassigned['parent_reassignment_similarity'].quantile(0.10)),
        'p25_similarity': float(reassigned['parent_reassignment_similarity'].quantile(0.25)),
        'low_similarity_notes': int(reassigned['parent_reassignment_similarity'].lt(threshold).sum()),
        'low_similarity_share': float(reassigned['parent_reassignment_similarity'].lt(threshold).mean()),
    }])
    by_topic = (
        reassigned.assign(low_similarity=reassigned['parent_reassignment_similarity'].lt(threshold))
        .groupby(['topic', 'topic_label'], dropna=False)
        .agg(
            reassigned_outlier_notes=('noteId', 'nunique'),
            mean_similarity=('parent_reassignment_similarity', 'mean'),
            median_similarity=('parent_reassignment_similarity', 'median'),
            low_similarity_notes=('low_similarity', 'sum'),
        )
        .reset_index()
    )
    by_topic['low_similarity_threshold'] = threshold
    by_topic['low_similarity_share'] = by_topic['low_similarity_notes'] / by_topic['reassigned_outlier_notes'].clip(lower=1)
    by_topic.insert(0, 'scope', 'by_parent_topic')
    return pd.concat([summary, by_topic], ignore_index=True, sort=False)


topic_parent_reassignment_sensitivity = build_reassignment_sensitivity(
    topic_parent_notes_reassigned,
    LOW_REASSIGNMENT_SIMILARITY_THRESHOLD,
)
print('Outlier reassignment sensitivity')
display(topic_parent_reassignment_sensitivity.head(40))

if not topic_parent_reassignment_sensitivity.empty and 'scope' in topic_parent_reassignment_sensitivity.columns:
    plot_low = topic_parent_reassignment_sensitivity[topic_parent_reassignment_sensitivity['scope'].eq('by_parent_topic')].copy()
    if not plot_low.empty:
        plot_low = plot_low.sort_values('low_similarity_notes', ascending=False).head(15)
        fig, ax = plt.subplots(figsize=(11, max(4, 0.35 * len(plot_low))))
        plot_low['plot_label'] = plot_low['topic'].astype(str) + ' | ' + plot_low['topic_label'].astype(str)
        sns.barplot(data=plot_low.sort_values('low_similarity_notes'), x='low_similarity_notes', y='plot_label', color='#B07AA1', ax=ax)
        ax.set_title(f'Low-confidence outlier reassignments (< {LOW_REASSIGNMENT_SIMILARITY_THRESHOLD:.2f})')
        ax.set_xlabel('Notes')
        ax.set_ylabel('Parent topic')
        plt.tight_layout()
        plt.show()


## Super Parent Reduction

This section reduces the 29 reassigned parent topics to a compact model-generated super-parent layer. The clustering unit is the parent-topic centroid, not individual notes. This keeps the 29-topic parent layer audit-friendly while producing a 5-6 topic narrative layer for paper-level summaries.

In [ ]:
def build_super_parent_layer(
    parent_notes_reassigned: pd.DataFrame,
    target_super_parent_topics: int,
    linkage: str = 'ward',
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    from sentence_transformers import SentenceTransformer
    from sklearn.cluster import AgglomerativeClustering

    if parent_notes_reassigned is None or parent_notes_reassigned.empty:
        raise ValueError('topic_parent_notes_reassigned is required for super-parent reduction.')
    if parent_notes_reassigned['topic'].eq(-1).any():
        raise ValueError('Super-parent reduction expects reassigned parent notes without topic = -1 outliers.')

    def topic_label_column(df: pd.DataFrame) -> str:
        if 'topic_display_label' in df.columns:
            return 'topic_display_label'
        if 'topic_label' in df.columns:
            return 'topic_label'
        return 'Name'

    parent_topic_col = 'topic'
    parent_label_col = topic_label_column(parent_notes_reassigned)
    parent_topics = sorted(parent_notes_reassigned[parent_topic_col].dropna().unique())
    if target_super_parent_topics >= len(parent_topics):
        raise ValueError(
            f'target_super_parent_topics={target_super_parent_topics} must be smaller than parent topics={len(parent_topics)}.'
        )

    embedder = SentenceTransformer(reassignment_embedding_model_name)
    texts = parent_notes_reassigned['summary'].astype(str).tolist()
    embeddings = _l2_normalize(embedder.encode(texts, show_progress_bar=True))
    notes = parent_notes_reassigned.reset_index(drop=True).copy()

    parent_centroids = []
    parent_rows = []
    for parent_topic in parent_topics:
        topic_positions = notes.index[notes[parent_topic_col].eq(parent_topic)].to_numpy()
        centroid = embeddings[topic_positions].mean(axis=0, keepdims=True)
        centroid = _l2_normalize(centroid)[0]
        parent_centroids.append(centroid)
        parent_label = notes.loc[topic_positions, parent_label_col].mode(dropna=True)
        parent_rows.append({
            'parent_topic': parent_topic,
            'parent_topic_label': parent_label.iloc[0] if not parent_label.empty else str(parent_topic),
            'parent_notes': len(topic_positions),
            'parent_reassigned_outlier_notes': int(notes.loc[topic_positions, 'reassigned_from_outlier'].sum())
            if 'reassigned_from_outlier' in notes.columns else 0,
        })

    parent_centroids = np.vstack(parent_centroids)
    clustering_kwargs = {'n_clusters': target_super_parent_topics, 'linkage': linkage}
    if linkage != 'ward':
        clustering_kwargs['metric'] = 'cosine'
    clusterer = AgglomerativeClustering(**clustering_kwargs)
    raw_super_labels = clusterer.fit_predict(parent_centroids)

    parent_crosswalk = pd.DataFrame(parent_rows)
    parent_crosswalk['raw_super_parent_topic'] = raw_super_labels

    # Stable ids: largest super-parent gets 0, then descending size.
    raw_sizes = (
        parent_crosswalk.groupby('raw_super_parent_topic')['parent_notes']
        .sum()
        .sort_values(ascending=False)
    )
    raw_to_stable = {raw_topic: idx for idx, raw_topic in enumerate(raw_sizes.index.tolist())}
    parent_crosswalk['super_parent_topic'] = parent_crosswalk['raw_super_parent_topic'].map(raw_to_stable).astype(int)

    def super_label(group: pd.DataFrame, n: int = 4) -> str:
        top = group.sort_values('parent_notes', ascending=False).head(n)
        labels = []
        for label in top['parent_topic_label'].astype(str):
            first_terms = ' '.join(label.split()[:2])
            if first_terms and first_terms not in labels:
                labels.append(first_terms)
        return ' / '.join(labels) if labels else f'Super Parent {int(group.name)}'

    super_labels = (
        parent_crosswalk.groupby('super_parent_topic')
        .apply(super_label)
        .rename('super_parent_label')
        .reset_index()
    )
    parent_crosswalk = parent_crosswalk.merge(super_labels, on='super_parent_topic', how='left')
    parent_crosswalk = parent_crosswalk.sort_values(['super_parent_topic', 'parent_notes'], ascending=[True, False])

    super_notes = notes.merge(
        parent_crosswalk[[
            'parent_topic',
            'parent_topic_label',
            'super_parent_topic',
            'super_parent_label',
        ]],
        left_on='topic',
        right_on='parent_topic',
        how='left',
    )
    super_notes['parent_topic'] = super_notes['topic']
    super_notes['parent_topic_label'] = super_notes[parent_label_col]
    super_notes['topic'] = super_notes['super_parent_topic']
    super_notes['Name'] = super_notes['super_parent_label']
    super_notes['topic_label'] = super_notes['super_parent_label']
    super_notes['topic_display_label'] = super_notes['super_parent_label']
    super_notes['super_parent_reduction_method'] = (
        f'parent_centroid_agglomerative_{linkage}_{target_super_parent_topics}'
    )

    super_stats = build_topic_cluster_stats(super_notes)
    super_stats = add_topic_display_labels(super_stats)
    super_stats = super_stats.rename(columns={
        'topic': 'super_parent_topic',
        'Name': 'super_parent_Name',
        'topic_label': 'super_parent_label',
        'topic_display_label': 'super_parent_display_label',
    })
    parent_counts = (
        parent_crosswalk.groupby('super_parent_topic')
        .agg(parent_topics=('parent_topic', 'nunique'))
        .reset_index()
    )
    super_stats = super_stats.merge(parent_counts, on='super_parent_topic', how='left')

    diagnostics = pd.DataFrame([{
        'input_parent_topics': len(parent_topics),
        'target_super_parent_topics': target_super_parent_topics,
        'final_super_parent_topics': parent_crosswalk['super_parent_topic'].nunique(),
        'input_notes': len(parent_notes_reassigned),
        'output_notes': len(super_notes),
        'remaining_outlier_notes': int(super_notes['topic'].eq(-1).sum()),
        'median_super_parent_size': float(super_notes.groupby('topic')['noteId'].nunique().median()),
        'max_super_parent_size': int(super_notes.groupby('topic')['noteId'].nunique().max()),
        'linkage': linkage,
        'embedding_model_name': reassignment_embedding_model_name,
    }])

    return super_notes, super_stats, parent_crosswalk, diagnostics


def _build_parent_centroids_for_super_parent(parent_notes_reassigned: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    from sentence_transformers import SentenceTransformer

    def topic_label_column(df: pd.DataFrame) -> str:
        if 'topic_display_label' in df.columns:
            return 'topic_display_label'
        if 'topic_label' in df.columns:
            return 'topic_label'
        return 'Name'

    notes = parent_notes_reassigned.reset_index(drop=True).copy()
    parent_label_col = topic_label_column(notes)
    embedder = SentenceTransformer(reassignment_embedding_model_name)
    embeddings = _l2_normalize(embedder.encode(notes['summary'].astype(str).tolist(), show_progress_bar=True))
    parent_rows = []
    centroids = []
    for parent_topic in sorted(notes['topic'].dropna().unique()):
        positions = notes.index[notes['topic'].eq(parent_topic)].to_numpy()
        centroid = _l2_normalize(embeddings[positions].mean(axis=0, keepdims=True))[0]
        centroids.append(centroid)
        label = notes.loc[positions, parent_label_col].mode(dropna=True)
        parent_rows.append({
            'parent_topic': parent_topic,
            'parent_topic_label': label.iloc[0] if not label.empty else str(parent_topic),
            'parent_notes': int(len(positions)),
            'parent_reassigned_outlier_notes': int(notes.loc[positions, 'reassigned_from_outlier'].sum())
            if 'reassigned_from_outlier' in notes.columns else 0,
        })
    return pd.DataFrame(parent_rows), np.vstack(centroids), embeddings


def _cluster_parent_centroids(parent_rows: pd.DataFrame, centroids: np.ndarray, k: int, linkage: str) -> pd.DataFrame:
    from sklearn.cluster import AgglomerativeClustering

    clustering_kwargs = {'n_clusters': k, 'linkage': linkage}
    if linkage != 'ward':
        clustering_kwargs['metric'] = 'cosine'
    raw_labels = AgglomerativeClustering(**clustering_kwargs).fit_predict(centroids)
    crosswalk = parent_rows.copy()
    crosswalk['raw_super_parent_topic'] = raw_labels
    raw_sizes = crosswalk.groupby('raw_super_parent_topic')['parent_notes'].sum().sort_values(ascending=False)
    raw_to_stable = {raw_topic: idx for idx, raw_topic in enumerate(raw_sizes.index.tolist())}
    crosswalk['super_parent_topic'] = crosswalk['raw_super_parent_topic'].map(raw_to_stable).astype(int)

    def super_label(group: pd.DataFrame, n: int = 4) -> str:
        top = group.sort_values('parent_notes', ascending=False).head(n)
        labels = []
        for label in top['parent_topic_label'].astype(str):
            first_terms = ' '.join(label.split()[:2])
            if first_terms and first_terms not in labels:
                labels.append(first_terms)
        return ' / '.join(labels) if labels else f'Super Parent {int(group.name)}'

    labels = crosswalk.groupby('super_parent_topic').apply(super_label).rename('super_parent_label').reset_index()
    return crosswalk.merge(labels, on='super_parent_topic', how='left')


def evaluate_super_parent_candidates(
    parent_rows: pd.DataFrame,
    centroids: np.ndarray,
    candidate_ks: list[int],
    linkage: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    sim = centroids @ centroids.T
    summary_rows = []
    detail_rows = []
    total_notes = parent_rows['parent_notes'].sum()
    for k in candidate_ks:
        if k >= len(parent_rows):
            continue
        cw = _cluster_parent_centroids(parent_rows, centroids, k=k, linkage=linkage)
        for super_topic, group in cw.groupby('super_parent_topic'):
            idx = group.index.to_numpy()
            if len(idx) > 1:
                pairwise = sim[np.ix_(idx, idx)][np.triu_indices(len(idx), k=1)]
                mean_sim = float(pairwise.mean())
                min_sim = float(pairwise.min())
            else:
                mean_sim = 1.0
                min_sim = 1.0
            parent_labels = ' / '.join(group.sort_values('parent_notes', ascending=False)['parent_topic_label'].astype(str).head(6))
            detail_rows.append({
                'candidate_k': k,
                'super_parent_topic': int(super_topic),
                'super_parent_label': group['super_parent_label'].iloc[0],
                'notes': int(group['parent_notes'].sum()),
                'share': float(group['parent_notes'].sum() / max(total_notes, 1)),
                'parent_topics': int(group['parent_topic'].nunique()),
                'mean_parent_similarity': mean_sim,
                'min_parent_similarity': min_sim,
                'parent_topic_labels': parent_labels,
            })
        d = pd.DataFrame([r for r in detail_rows if r['candidate_k'] == k])
        summary_rows.append({
            'candidate_k': k,
            'super_parent_topics': int(d['super_parent_topic'].nunique()),
            'largest_super_parent_share': float(d['share'].max()),
            'max_parent_topics_in_super_parent': int(d['parent_topics'].max()),
            'singleton_super_parent_count': int(d['parent_topics'].eq(1).sum()),
            'mean_within_super_parent_similarity': float(d['mean_parent_similarity'].replace(1.0, np.nan).mean()),
            'min_within_super_parent_similarity': float(d.loc[d['parent_topics'].gt(1), 'min_parent_similarity'].min()) if d['parent_topics'].gt(1).any() else 1.0,
            'passes_largest_share_gate': bool(d['share'].max() <= MAX_SUPER_PARENT_SHARE),
            'passes_parent_count_gate': bool(d['parent_topics'].max() <= MAX_PARENTS_PER_SUPER_PARENT),
        })
    summary = pd.DataFrame(summary_rows)
    if not summary.empty:
        summary['passes_quality_gate'] = (
            summary['passes_largest_share_gate']
            & summary['passes_parent_count_gate']
            & summary['mean_within_super_parent_similarity'].fillna(1).ge(MIN_MEAN_WITHIN_SUPER_PARENT_SIMILARITY)
        )
    return summary, pd.DataFrame(detail_rows)


super_parent_input_notes = topic_parent_notes_reassigned
if EXCLUDE_META_DOMINANT_TOPICS_FROM_SUPER_PARENT and 'is_meta_dominant_topic' in super_parent_input_notes.columns:
    excluded_meta_topics = sorted(
        super_parent_input_notes.loc[super_parent_input_notes['is_meta_dominant_topic'], 'topic']
        .dropna()
        .unique()
        .tolist()
    )
    super_parent_input_notes = super_parent_input_notes[~super_parent_input_notes['is_meta_dominant_topic']].copy()
    print(f'Excluding meta-dominant parent topics from super-parent reduction: {excluded_meta_topics}')

parent_rows_for_super, parent_centroids_for_super, _ = _build_parent_centroids_for_super_parent(super_parent_input_notes)
topic_super_parent_candidate_quality, topic_super_parent_candidate_crosswalk = evaluate_super_parent_candidates(
    parent_rows_for_super,
    parent_centroids_for_super,
    candidate_super_parent_topics,
    super_parent_linkage,
)

if AUTO_SELECT_SUPER_PARENT_TOPIC_COUNT and not topic_super_parent_candidate_quality.empty:
    passing = topic_super_parent_candidate_quality[topic_super_parent_candidate_quality['passes_quality_gate']].copy()
    if not passing.empty:
        selected_super_parent_topics = int(passing.sort_values(['candidate_k']).iloc[0]['candidate_k'])
    else:
        selected_super_parent_topics = int(
            topic_super_parent_candidate_quality
            .sort_values(['largest_super_parent_share', 'max_parent_topics_in_super_parent', 'candidate_k'])
            .iloc[0]['candidate_k']
        )
else:
    selected_super_parent_topics = target_super_parent_topics

print(f'Selected super-parent topic count: {selected_super_parent_topics}')
display(topic_super_parent_candidate_quality)
display(topic_super_parent_candidate_crosswalk.sort_values(['candidate_k', 'super_parent_topic', 'notes'], ascending=[True, True, False]).head(80))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
sns.lineplot(data=topic_super_parent_candidate_quality, x='candidate_k', y='largest_super_parent_share', marker='o', ax=axes[0])
axes[0].axhline(MAX_SUPER_PARENT_SHARE, color='crimson', ls='--', lw=1)
axes[0].set_title('Largest super-parent share')
axes[0].set_xlabel('Candidate k')
axes[0].set_ylabel('Share')
sns.lineplot(data=topic_super_parent_candidate_quality, x='candidate_k', y='max_parent_topics_in_super_parent', marker='o', ax=axes[1])
axes[1].axhline(MAX_PARENTS_PER_SUPER_PARENT, color='crimson', ls='--', lw=1)
axes[1].set_title('Max parent topics grouped')
axes[1].set_xlabel('Candidate k')
axes[1].set_ylabel('Parent topics')
sns.lineplot(data=topic_super_parent_candidate_quality, x='candidate_k', y='mean_within_super_parent_similarity', marker='o', ax=axes[2])
axes[2].axhline(MIN_MEAN_WITHIN_SUPER_PARENT_SIMILARITY, color='crimson', ls='--', lw=1)
axes[2].set_title('Mean within-group parent similarity')
axes[2].set_xlabel('Candidate k')
axes[2].set_ylabel('Cosine similarity')
plt.tight_layout()
plt.show()

topic_super_parent_notes, topic_super_parent_cluster_stats, topic_super_parent_crosswalk, topic_super_parent_diagnostics = build_super_parent_layer(
    super_parent_input_notes,
    target_super_parent_topics=selected_super_parent_topics,
    linkage=super_parent_linkage,
)

print('Super-parent diagnostics')
display(topic_super_parent_diagnostics)
print('Parent topic -> super-parent crosswalk')
display(topic_super_parent_crosswalk)
print('Super-parent cluster stats')
display(topic_super_parent_cluster_stats.sort_values('notes', ascending=False))

## Result Review & Topic Distributions

These cells make the parent layer inspectable: parent topic size distribution, fine-vs-parent compression diagnostics, rank-size comparison, dominant fine-topic mapping, and a crosswalk heatmap.

In [ ]:
def _topic_label_col(df: pd.DataFrame) -> str:
    if 'topic_display_label' in df.columns:
        return 'topic_display_label'
    if 'topic_label' in df.columns:
        return 'topic_label'
    return 'Name'

parent_label_col = _topic_label_col(parent_notes)
parent_notes_for_review = parent_notes
if EXCLUDE_META_DOMINANT_TOPICS_FROM_REVIEW_PLOTS and 'is_meta_dominant_topic' in parent_notes_for_review.columns:
    excluded_review_topics = sorted(parent_notes_for_review.loc[parent_notes_for_review['is_meta_dominant_topic'], 'topic'].dropna().unique().tolist())
    parent_notes_for_review = parent_notes_for_review[~parent_notes_for_review['is_meta_dominant_topic']].copy()
    print(f'Excluding meta-dominant parent topics from review plots: {excluded_review_topics}')

parent_topic_distribution = (
    parent_notes_for_review.groupby(['topic', parent_label_col], dropna=False)
    .agg(
        notes=('noteId', 'nunique'),
        avg_votes=('total_votes', 'mean'),
        avg_abs_gap=('abs_gap', 'mean'),
    )
    .reset_index()
    .rename(columns={parent_label_col: 'topic_label'})
)
parent_topic_distribution['share'] = parent_topic_distribution['notes'] / parent_topic_distribution['notes'].sum()
parent_topic_distribution = parent_topic_distribution.sort_values('notes', ascending=False)
parent_topic_distribution['share_pct'] = (100 * parent_topic_distribution['share']).round(2)

print('Parent topic distribution')
display(parent_topic_distribution.head(40))

plot_df = parent_topic_distribution.copy()
plot_df['plot_label'] = plot_df['topic'].astype(str) + ' | ' + plot_df['topic_label'].astype(str)
plot_df = plot_df.sort_values('notes', ascending=True)
fig, ax = plt.subplots(figsize=(12, max(7, 0.32 * len(plot_df))))
sns.barplot(data=plot_df, x='notes', y='plot_label', color='#4C78A8', ax=ax)
ax.set_title('Parent Topic Size Distribution')
ax.set_xlabel('Notes')
ax.set_ylabel('Parent topic')
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3, fontsize=8)
plt.tight_layout()
plt.show()

if CREATE_REASSIGNED_PARENT_NOTES and topic_parent_notes_reassigned is not None and not topic_parent_notes_reassigned.empty:
    reassigned_label_col = _topic_label_col(topic_parent_notes_reassigned)
    parent_reassigned_topic_distribution = (
        topic_parent_notes_reassigned[~topic_parent_notes_reassigned.get('is_meta_dominant_topic', False)].groupby(['topic', reassigned_label_col], dropna=False)
        .agg(
            notes=('noteId', 'nunique'),
            avg_votes=('total_votes', 'mean'),
            avg_abs_gap=('abs_gap', 'mean'),
            reassigned_outlier_notes=('reassigned_from_outlier', 'sum'),
        )
        .reset_index()
        .rename(columns={reassigned_label_col: 'topic_label'})
    )
    parent_reassigned_topic_distribution['share'] = parent_reassigned_topic_distribution['notes'] / parent_reassigned_topic_distribution['notes'].sum()
    parent_reassigned_topic_distribution = parent_reassigned_topic_distribution.sort_values('notes', ascending=False)
    parent_reassigned_topic_distribution['share_pct'] = (100 * parent_reassigned_topic_distribution['share']).round(2)

    print('Parent topic distribution after outlier reassignment')
    display(parent_reassigned_topic_distribution.head(40))

    reassigned_plot_df = parent_reassigned_topic_distribution.copy()
    reassigned_plot_df['plot_label'] = reassigned_plot_df['topic'].astype(str) + ' | ' + reassigned_plot_df['topic_label'].astype(str)
    reassigned_plot_df = reassigned_plot_df.sort_values('notes', ascending=True)
    fig, ax = plt.subplots(figsize=(12, max(7, 0.32 * len(reassigned_plot_df))))
    sns.barplot(data=reassigned_plot_df, x='notes', y='plot_label', color='#59A14F', ax=ax)
    ax.set_title('Parent Topic Size Distribution After Outlier Reassignment')
    ax.set_xlabel('Notes')
    ax.set_ylabel('Parent topic')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.0f', padding=3, fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    parent_reassigned_topic_distribution = pd.DataFrame()

super_parent_distribution = (
    topic_super_parent_notes.groupby(['topic', 'topic_label'], dropna=False)
    .agg(
        notes=('noteId', 'nunique'),
        parent_topics=('parent_topic', 'nunique'),
        avg_votes=('total_votes', 'mean'),
        avg_abs_gap=('abs_gap', 'mean'),
        reassigned_outlier_notes=('reassigned_from_outlier', 'sum'),
    )
    .reset_index()
    .sort_values('notes', ascending=False)
)
super_parent_distribution['share'] = super_parent_distribution['notes'] / super_parent_distribution['notes'].sum()
super_parent_distribution['share_pct'] = (100 * super_parent_distribution['share']).round(2)

print('Super-parent topic distribution')
display(super_parent_distribution)

super_plot_df = super_parent_distribution.sort_values('notes', ascending=True).copy()
super_plot_df['plot_label'] = super_plot_df['topic'].astype(str) + ' | ' + super_plot_df['topic_label'].astype(str)
fig, ax = plt.subplots(figsize=(11, max(4.5, 0.5 * len(super_plot_df))))
sns.barplot(data=super_plot_df, x='notes', y='plot_label', color='#F28E2B', ax=ax)
ax.set_title('Super-Parent Topic Size Distribution')
ax.set_xlabel('Notes')
ax.set_ylabel('Super-parent topic')
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
def build_layer_distribution(notes: pd.DataFrame, layer_name: str) -> pd.DataFrame:
    label_col = _topic_label_col(notes)
    dist = (
        notes.groupby(['topic', label_col], dropna=False)
        .agg(notes=('noteId', 'nunique'))
        .reset_index()
        .rename(columns={label_col: 'topic_label'})
        .sort_values('notes', ascending=False)
    )
    dist['layer'] = layer_name
    dist['rank'] = range(1, len(dist) + 1)
    dist['share'] = dist['notes'] / dist['notes'].sum()
    return dist

if fine_topic_notes is not None:
    fine_distribution = build_layer_distribution(fine_topic_notes, 'fine BERTopic')
    parent_distribution_for_compare = build_layer_distribution(parent_notes, 'parent BERTopic strict')
    comparison_distributions = [fine_distribution, parent_distribution_for_compare]
    if CREATE_REASSIGNED_PARENT_NOTES and topic_parent_notes_reassigned is not None and not topic_parent_notes_reassigned.empty:
        parent_reassigned_distribution_for_compare = build_layer_distribution(topic_parent_notes_reassigned, 'parent BERTopic reassigned')
        comparison_distributions.append(parent_reassigned_distribution_for_compare)
    super_parent_distribution_for_compare = build_layer_distribution(topic_super_parent_notes, 'super-parent BERTopic')
    comparison_distributions.append(super_parent_distribution_for_compare)

    def layer_summary(dist: pd.DataFrame, layer_name: str) -> dict:
        non_outlier = dist[dist['topic'] != -1]
        outlier_notes = int(dist.loc[dist['topic'].eq(-1), 'notes'].sum())
        return {
            'layer': layer_name,
            'topics_including_outlier': dist['topic'].nunique(),
            'non_outlier_topics': non_outlier['topic'].nunique(),
            'outlier_notes': outlier_notes,
            'outlier_share': outlier_notes / max(int(dist['notes'].sum()), 1),
            'median_non_outlier_topic_size': float(non_outlier['notes'].median()) if not non_outlier.empty else np.nan,
            'max_non_outlier_topic_size': int(non_outlier['notes'].max()) if not non_outlier.empty else 0,
        }

    fine_parent_comparison = pd.DataFrame([
        layer_summary(dist, dist['layer'].iloc[0])
        for dist in comparison_distributions
    ])
    fine_parent_comparison['outlier_share_pct'] = (100 * fine_parent_comparison['outlier_share']).round(2)
    print('Fine vs parent topic compression')
    display(fine_parent_comparison)

    rank_df = pd.concat(comparison_distributions, ignore_index=True)
    fig, ax = plt.subplots(figsize=(10, 5.5))
    sns.lineplot(data=rank_df, x='rank', y='notes', hue='layer', marker='o', ax=ax)
    ax.set_yscale('log')
    ax.set_title('Topic Size Rank Comparison')
    ax.set_xlabel('Topic rank by size')
    ax.set_ylabel('Notes, log scale')
    plt.tight_layout()
    plt.show()
else:
    fine_distribution = pd.DataFrame()
    fine_parent_comparison = pd.DataFrame()
    print('No fine topic_notes.parquet found; skipping fine-vs-parent distribution comparison.')

In [ ]:
if fine_topic_notes is not None and not topic_parent_crosswalk.empty:
    dominant_crosswalk = topic_parent_crosswalk[topic_parent_crosswalk['is_dominant_parent']].copy()
    dominant_crosswalk = dominant_crosswalk.sort_values(['parent_topic', 'notes'], ascending=[True, False])

    parent_absorbs = (
        dominant_crosswalk[dominant_crosswalk['fine_topic'] != -1]
        .groupby(['parent_topic', 'parent_topic_label'])
        .agg(
            dominant_fine_topics=('fine_topic', 'nunique'),
            notes_in_dominant_fine_topics=('notes', 'sum'),
        )
        .reset_index()
        .sort_values(['dominant_fine_topics', 'notes_in_dominant_fine_topics'], ascending=False)
    )

    def _top_fine_topics(group: pd.DataFrame, n: int = 6) -> str:
        top = group.sort_values('notes', ascending=False).head(n)
        return '; '.join(
            f"{row.fine_topic_label} ({int(row.notes)})"
            for row in top.itertuples(index=False)
        )

    top_fine_by_parent = (
        dominant_crosswalk[dominant_crosswalk['fine_topic'] != -1]
        .groupby(['parent_topic', 'parent_topic_label'], group_keys=False)
        .apply(_top_fine_topics)
        .reset_index(name='top_dominant_fine_topics')
    )
    parent_absorbs = parent_absorbs.merge(top_fine_by_parent, on=['parent_topic', 'parent_topic_label'], how='left')

    print('Which parent topics absorb the most fine topics?')
    display(parent_absorbs.head(30))

    print('Dominant parent assignment for each fine topic')
    display(
        dominant_crosswalk
        .sort_values(['fine_topic'])
        .loc[:, ['fine_topic', 'fine_topic_label', 'parent_topic', 'parent_topic_label', 'notes', 'share_within_fine_topic']]
        .head(80)
    )

    heatmap_source = topic_parent_crosswalk[topic_parent_crosswalk['fine_topic'] != -1].copy()
    top_fine_topics = (
        heatmap_source.groupby(['fine_topic', 'fine_topic_label'])['notes']
        .sum()
        .sort_values(ascending=False)
        .head(25)
        .reset_index()['fine_topic']
    )
    heatmap_source = heatmap_source[heatmap_source['fine_topic'].isin(top_fine_topics)]
    heatmap_matrix = heatmap_source.pivot_table(
        index='fine_topic_label',
        columns='parent_topic_label',
        values='share_within_fine_topic',
        aggfunc='sum',
        fill_value=0,
    )
    keep_parent_cols = heatmap_matrix.sum(axis=0).sort_values(ascending=False).head(18).index
    heatmap_matrix = heatmap_matrix.loc[:, keep_parent_cols]
    ordered_rows = heatmap_matrix.max(axis=1).sort_values(ascending=True).index
    heatmap_matrix = heatmap_matrix.loc[ordered_rows]

    fig, ax = plt.subplots(figsize=(15, max(7, 0.36 * len(heatmap_matrix))))
    sns.heatmap(heatmap_matrix, cmap='viridis', vmin=0, vmax=1, linewidths=0.25, ax=ax)
    ax.set_title('Fine Topic to Parent Topic Crosswalk: Share Within Fine Topic')
    ax.set_xlabel('Parent topic')
    ax.set_ylabel('Fine topic')
    plt.tight_layout()
    plt.show()
else:
    dominant_crosswalk = pd.DataFrame()
    parent_absorbs = pd.DataFrame()
    print('No crosswalk available; skipping parent/fine crosswalk review.')

## Persist New Artifacts

In [ ]:
FORBIDDEN_TOPIC_OUTPUTS = {
    'topic_notes.parquet',
    'topic_cluster_stats.parquet',
    'topic_exemplars.parquet',
    'topic_salience.parquet',
    'topic_salience_pivot.parquet',
    'topic_rescue_stats.parquet',
    'topic_strategy_summary.parquet',
    'topic_strategy_pivot.parquet',
    'topic_selection_overlap.parquet',
}

PERSIST_TABLES = {
    'notes': parent_notes,
    'notes_reassigned': topic_parent_notes_reassigned,
    'cluster_stats': parent_cluster_stats,
    'exemplars': parent_exemplars,
    'crosswalk': topic_parent_crosswalk,
    'diagnostics': topic_parent_diagnostics,
}

PARENT_AUDIT_TABLES = {
    'initial_topics': parent_initial_topics,
    'initial_topic_terms': parent_initial_topic_terms,
    'meta_terms': topic_parent_meta_terms,
    'meta_topic_diagnostics': topic_parent_meta_topic_diagnostics,
    'reassignment_sensitivity': topic_parent_reassignment_sensitivity,
}

SUPER_PARENT_AUDIT_TABLES = {
    'candidate_quality': topic_super_parent_candidate_quality,
    'candidate_crosswalk': topic_super_parent_candidate_crosswalk,
}

SUPER_PERSIST_TABLES = {
    'notes': topic_super_parent_notes,
    'cluster_stats': topic_super_parent_cluster_stats,
    'crosswalk': topic_super_parent_crosswalk,
    'diagnostics': topic_super_parent_diagnostics,
}

def assert_parent_output_path(path: Path) -> None:
    if path.parent.resolve() != PROCESSED_DIR.resolve():
        raise ValueError(f'Parent output must stay inside {PROCESSED_DIR}: {path}')
    if path.name in FORBIDDEN_TOPIC_OUTPUTS:
        raise ValueError(f'Refusing to write existing fine-topic artifact: {path.name}')
    if not path.name.startswith('topic_parent_') or path.suffix != '.parquet':
        raise ValueError(f'Refusing non-parent output path: {path}')

def assert_super_parent_output_path(path: Path) -> None:
    if path.parent.resolve() != PROCESSED_DIR.resolve():
        raise ValueError(f'Super-parent output must stay inside {PROCESSED_DIR}: {path}')
    if path.name in FORBIDDEN_TOPIC_OUTPUTS:
        raise ValueError(f'Refusing to write existing fine-topic artifact: {path.name}')
    if not path.name.startswith('topic_super_parent_') or path.suffix != '.parquet':
        raise ValueError(f'Refusing non-super-parent output path: {path}')

def save_or_skip_parent_table(name: str, df: pd.DataFrame, path: Path) -> dict:
    assert_parent_output_path(path)
    if path.exists() and not ALLOW_OVERWRITE_PARENT_OUTPUTS:
        if SKIP_EXISTING_PARENT_OUTPUTS:
            return {'artifact': name, 'status': 'skipped_existing', 'path': str(path), 'rows': len(df)}
        raise FileExistsError(
            f'{path} already exists. Set ALLOW_OVERWRITE_PARENT_OUTPUTS = True to replace only topic_parent_* outputs, '
            'or set SKIP_EXISTING_PARENT_OUTPUTS = True to keep existing files and continue.'
        )
    save_table(df, path)
    return {
        'artifact': name,
        'status': 'overwritten' if path.exists() and ALLOW_OVERWRITE_PARENT_OUTPUTS else 'written',
        'path': str(path),
        'rows': len(df),
    }


def save_or_skip_super_parent_table(name: str, df: pd.DataFrame, path: Path) -> dict:
    assert_super_parent_output_path(path)
    if path.exists() and not ALLOW_OVERWRITE_PARENT_OUTPUTS:
        if SKIP_EXISTING_PARENT_OUTPUTS:
            return {'artifact': f'super_{name}', 'status': 'skipped_existing', 'path': str(path), 'rows': len(df)}
        raise FileExistsError(
            f'{path} already exists. Set ALLOW_OVERWRITE_PARENT_OUTPUTS = True to replace only topic_super_parent_* outputs, '
            'or set SKIP_EXISTING_PARENT_OUTPUTS = True to keep existing files and continue.'
        )
    save_table(df, path)
    return {
        'artifact': f'super_{name}',
        'status': 'overwritten' if path.exists() and ALLOW_OVERWRITE_PARENT_OUTPUTS else 'written',
        'path': str(path),
        'rows': len(df),
    }

parent_persist_results = [
    save_or_skip_parent_table(name, PERSIST_TABLES[name], PARENT_OUTPUTS[name])
    for name in PARENT_OUTPUTS
]
parent_audit_persist_results = [
    save_or_skip_parent_table(name, PARENT_AUDIT_TABLES[name], PARENT_AUDIT_OUTPUTS[name])
    for name in PARENT_AUDIT_OUTPUTS
    if PARENT_AUDIT_TABLES.get(name) is not None and not PARENT_AUDIT_TABLES[name].empty
]

super_parent_audit_persist_results = [
    save_or_skip_super_parent_table(name, SUPER_PARENT_AUDIT_TABLES[name], SUPER_PARENT_AUDIT_OUTPUTS[name])
    for name in SUPER_PARENT_AUDIT_OUTPUTS
    if SUPER_PARENT_AUDIT_TABLES.get(name) is not None and not SUPER_PARENT_AUDIT_TABLES[name].empty
]

super_parent_persist_results = [
    save_or_skip_super_parent_table(name, SUPER_PERSIST_TABLES[name], SUPER_PARENT_OUTPUTS[name])
    for name in SUPER_PARENT_OUTPUTS
]
persist_results = pd.DataFrame(parent_persist_results + parent_audit_persist_results + super_parent_audit_persist_results + super_parent_persist_results)
display(persist_results)

## Final Topic Modeling Pipeline Summary

Technical summary of the current run:

1. **Input layer:** `scores.parquet` is converted to one note-level row per Community Note summary.
2. **Initial BERTopic audit:** BERTopic is first fit normally with SentenceTransformer embeddings, UMAP, HDBSCAN, and c-TF-IDF. The initial unreduced topic list and topic terms are saved for audit.
3. **Meta-term representation cleanup:** Community Notes process terms are removed from topic representations, not from the corpus. This prevents words such as `note`, `notes`, `community`, `NNN`, `comment(s)`, `section`, `Birdwatch`, and `rated/rating` from dominating topic labels.
4. **Parent-topic layer:** BERTopic is reduced to the parent layer (`target_parent_topics = 30`), producing 29 non-outlier parent topics in the current run.
5. **Meta-dominant topic flagging:** Parent topics whose summaries are mostly about no-note-needed / Community Notes governance disputes are flagged. The current run flags `Opinion Abusing Stop Stop` because `597 / 705 = 84.7%` of its notes match meta-governance patterns.
6. **Outlier reassignment:** HDBSCAN outliers are reassigned to the nearest parent-topic centroid. The strict original assignment is preserved in `original_parent_*` columns. A sensitivity table reports low-confidence reassignments below similarity `0.35`.
7. **Super-parent candidate evaluation:** Candidate super-parent counts `6`, `8`, `10`, and `12` are evaluated using parent-centroid clustering. The quality gate checks whether a candidate avoids overly dominant super topics, avoids grouping too many parent topics together, and maintains reasonable within-group parent similarity.
8. **Final super-parent layer:** `k=8` is selected because `k=6` fails both concentration and parent-count gates, while `k=8` is the smallest candidate that passes. Meta-dominant parent topics are excluded from this narrative layer.

Plain-language version:

1. We first let BERTopic discover topics normally.
2. We look at the raw topic words and remove words that are about the Community Notes system itself rather than the tweet content.
3. We keep all notes, but we mark topics that are mostly about “no note needed,” “use the comments,” or “don’t abuse Community Notes.”
4. We keep a detailed parent-topic layer for analysis.
5. We assign outlier notes to their nearest parent topic, but we also report which assignments are uncertain.
6. We try several possible numbers of broad topics instead of forcing six.
7. Six broad topics turned out to be too compressed, so the notebook selected eight broad topics.
8. The final setup is: `topic_parent_notes_reassigned.parquet` for detailed analysis and `topic_super_parent_notes.parquet` for paper-level narrative/visualization.
